## 原本pipeline

### 寫入資料 -> 切分航程 -> 異常航程判定 

In [1]:
# =========================
# main.py in Jupyter cell（整合版 - smart teleport + 港到港 + robust停泊分類）
# =========================

from pathlib import Path
import webbrowser

import numpy as np
import pandas as pd
from haversine import haversine

# 匯入自訂模組
from data_loader import load_and_preprocess
from voyage_splitter import (
    detect_stops,
    split_voyages,
    voyage_quality_checker,
    _safe_haversine,   # 小工具：改經度範圍後的 haversine
)
from visualization import visualize_voyages
from od_marker import assign_ports
from Trajectory_Recon import TrajectoryReconstructor
from data_quality import voyage_integrity_metrics


# ----------------------------
# 0. 基本設定
# ----------------------------
csv_path = Path(r"C:\Users\slab\Desktop") / "Slab Project" / "Stage1" / "data" / "477300400_VesselHistoryLineInfo.csv"
target_mmsi = 477300400

# 港口清單路徑：filtered_ports.csv，欄位為 unlocode, lat, lon
ports_csv = Path(r"C:\Users\slab\Desktop") / "Slab Project" / "Stage1" / "data" / "filtered_ports.csv"


# ----------------------------
# 0.5  移除雜訊點（teleport）- smart 迭代版
# ----------------------------

def remove_teleport_points(
    df: pd.DataFrame,
    speed_kn_th: float = 50.0,
    dt_max_sec: float = 3600.0,
    keep_report: bool = True
):
    """
    移除 AIS 瞬移點（teleport）
    - Rule A（三點檢查，中間點）：prev->cur 爆速 且 prev->next 合理 => 刪 cur
    - Rule B（開頭邊界點）：cur->next 爆速 且 next->next 正常 => 刪 cur
    - Rule C（結尾邊界點）：prev->cur 爆速 且 prev->prev 正常 => 刪 cur

    需要欄位：Timestamp, Lat, Long
    回傳：df_clean, report_df
    """
    d = df.sort_values("Timestamp").copy()

    # 避免你 df 本來就有 orig_index 造成重複欄位
    if "orig_index" in d.columns:
        d = d.rename(columns={"orig_index": "orig_index_old"})
    d = d.reset_index(drop=False).rename(columns={"index": "orig_index"})

    # prev/next
    d["Lat_prev"]   = d["Lat"].shift(1)
    d["Long_prev"]  = d["Long"].shift(1)
    d["T_prev"]     = d["Timestamp"].shift(1)

    d["Lat_next"]   = d["Lat"].shift(-1)
    d["Long_next"]  = d["Long"].shift(-1)
    d["T_next"]     = d["Timestamp"].shift(-1)

    d["Lat_next2"]  = d["Lat"].shift(-2)
    d["Long_next2"] = d["Long"].shift(-2)
    d["T_next2"]    = d["Timestamp"].shift(-2)

    d["Lat_prev2"]  = d["Lat"].shift(2)
    d["Long_prev2"] = d["Long"].shift(2)
    d["T_prev2"]    = d["Timestamp"].shift(2)

    # dt
    d["dt_prev_sec"]     = (d["Timestamp"] - d["T_prev"]).dt.total_seconds()
    d["dt_next_sec"]     = (d["T_next"] - d["Timestamp"]).dt.total_seconds()
    d["dt_prevnext_sec"] = (d["T_next"] - d["T_prev"]).dt.total_seconds()
    d["dt_next2_sec"]    = (d["T_next2"] - d["T_next"]).dt.total_seconds()
    d["dt_prev2_sec"]    = (d["T_prev"] - d["T_prev2"]).dt.total_seconds()

    # dist (km)
    def _dist_km(lat1, lon1, lat2, lon2):
        if np.any(pd.isna([lat1, lon1, lat2, lon2])):
            return np.nan
        return haversine((lat1, lon1), (lat2, lon2))

    d["dist_prev_km"]     = [_dist_km(a,b,c,e) for a,b,c,e in zip(d["Lat_prev"],  d["Long_prev"],  d["Lat"],      d["Long"])]
    d["dist_next_km"]     = [_dist_km(a,b,c,e) for a,b,c,e in zip(d["Lat"],       d["Long"],       d["Lat_next"], d["Long_next"])]
    d["dist_prevnext_km"] = [_dist_km(a,b,c,e) for a,b,c,e in zip(d["Lat_prev"],  d["Long_prev"],  d["Lat_next"], d["Long_next"])]

    d["dist_next2_km"]    = [_dist_km(a,b,c,e) for a,b,c,e in zip(d["Lat_next"],  d["Long_next"],  d["Lat_next2"], d["Long_next2"])]
    d["dist_prev2_km"]    = [_dist_km(a,b,c,e) for a,b,c,e in zip(d["Lat_prev2"], d["Long_prev2"], d["Lat_prev"],  d["Long_prev"])]

    # speed (kn)
    d["speed_prev_kn"]     = (d["dist_prev_km"] / (d["dt_prev_sec"] / 3600.0)) / 1.852
    d["speed_next_kn"]     = (d["dist_next_km"] / (d["dt_next_sec"] / 3600.0)) / 1.852
    d["speed_prevnext_kn"] = (d["dist_prevnext_km"] / (d["dt_prevnext_sec"] / 3600.0)) / 1.852
    d["speed_next2_kn"]    = (d["dist_next2_km"] / (d["dt_next2_sec"] / 3600.0)) / 1.852
    d["speed_prev2_kn"]    = (d["dist_prev2_km"] / (d["dt_prev2_sec"] / 3600.0)) / 1.852

    # ---------- Rule A: 三點檢查刪中間點 ----------
    ruleA = (
        (d["dt_prev_sec"] > 0) & (d["dt_prev_sec"] <= dt_max_sec) &
        (d["speed_prev_kn"] > speed_kn_th) &
        (d["speed_prevnext_kn"].fillna(np.inf) <= speed_kn_th)
    )

    # ---------- Rule B: 開頭邊界離群點（刪當前點） ----------
    ruleB = (
        (d["dt_next_sec"] > 0) & (d["dt_next_sec"] <= dt_max_sec) &
        (d["speed_next_kn"] > speed_kn_th) &
        (d["speed_next2_kn"].fillna(np.inf) <= speed_kn_th)
    )

    # ---------- Rule C: 結尾邊界離群點（刪當前點） ----------
    ruleC = (
        (d["dt_prev_sec"] > 0) & (d["dt_prev_sec"] <= dt_max_sec) &
        (d["speed_prev_kn"] > speed_kn_th) &
        (d["speed_prev2_kn"].fillna(np.inf) <= speed_kn_th)
    )

    to_drop = (ruleA | ruleB | ruleC).fillna(False)

    reason = np.where(ruleA, "A_midpoint",
             np.where(ruleB, "B_head_outlier",
             np.where(ruleC, "C_tail_outlier", "")))
    report = d.loc[to_drop, ["orig_index","Timestamp","Lat","Long","speed_prev_kn","speed_next_kn"]].copy()
    report["reason"] = np.array(reason)[to_drop.to_numpy()]

    drop_cols = [
        "Lat_prev","Long_prev","T_prev",
        "Lat_next","Long_next","T_next",
        "Lat_next2","Long_next2","T_next2",
        "Lat_prev2","Long_prev2","T_prev2",
        "dt_prev_sec","dt_next_sec","dt_prevnext_sec","dt_next2_sec","dt_prev2_sec",
        "dist_prev_km","dist_next_km","dist_prevnext_km","dist_next2_km","dist_prev2_km",
        "speed_prev_kn","speed_next_kn","speed_prevnext_kn","speed_next2_kn","speed_prev2_kn"
    ]
    drop_cols = [c for c in drop_cols if c in d.columns]

    df_clean = d.loc[~to_drop, :].drop(columns=drop_cols).reset_index(drop=True)

    if keep_report:
        return df_clean, report
    return df_clean, None


def has_teleport_edges(df: pd.DataFrame, speed_kn_th: float = 50.0, dt_max_sec: float = 3600.0) -> bool:
    if df.empty or len(df) < 2:
        return False

    d = df.sort_values("Timestamp")[["Timestamp","Lat","Long"]].copy()
    dt = d["Timestamp"].diff().dt.total_seconds()

    lat1 = d["Lat"].shift(1).to_numpy()
    lon1 = d["Long"].shift(1).to_numpy()
    lat2 = d["Lat"].to_numpy()
    lon2 = d["Long"].to_numpy()

    dist = np.full(len(d), np.nan, dtype=float)
    valid = (~np.isnan(lat1)) & (~np.isnan(lon1)) & (~np.isnan(lat2)) & (~np.isnan(lon2))
    idxs = np.where(valid)[0]
    for i in idxs:
        dist[i] = haversine((lat1[i], lon1[i]), (lat2[i], lon2[i]))  # km

    speed_kn = (dist / (dt.to_numpy() / 3600.0)) / 1.852
    mask = (dt.to_numpy() > 0) & (dt.to_numpy() <= dt_max_sec) & (speed_kn > speed_kn_th)
    return bool(np.nan_to_num(mask).any())


def remove_teleport_points_smart(
    df: pd.DataFrame,
    speed_kn_th: float = 50.0,
    dt_max_sec: float = 3600.0,
    max_iter: int = 8,
    verbose: bool = True
):
    d = df.copy()
    reports = []

    for it in range(max_iter):
        still_bad = has_teleport_edges(d, speed_kn_th=speed_kn_th, dt_max_sec=dt_max_sec)
        if not still_bad:
            if verbose:
                print(f"[teleport] stop at iter {it}: no suspect edges.")
            break

        d2, rep = remove_teleport_points(d, speed_kn_th=speed_kn_th, dt_max_sec=dt_max_sec, keep_report=True)
        n_drop = 0 if rep is None else len(rep)
        if verbose:
            print(f"[teleport] iter {it+1}: dropped {n_drop} points")

        if rep is None or n_drop == 0:
            break

        rep = rep.copy()
        rep["iter"] = it + 1
        reports.append(rep)
        d = d2

    report_all = pd.concat(reports, ignore_index=True) if reports else pd.DataFrame()
    return d, report_all


# ----------------------------
# 1. 載入 AIS & teleport 清洗 & 基礎停泊偵測
# ----------------------------
print("載入與前處理資料中...")
df = load_and_preprocess(csv_path, target_mmsi)
print(f"清理後資料筆數: {len(df)}")

print("移除瞬移(teleport)雜訊點（smart迭代：先掃描、有異常才跑下一輪）...")
df, teleport_report = remove_teleport_points_smart(
    df,
    speed_kn_th=50,
    dt_max_sec=3600,
    max_iter=8,
    verbose=True
)
print(f"刪除 teleport 點數量: {len(teleport_report)}")
print(f"清理後資料筆數(teleport 清除後): {len(df)}")

print("執行停泊判斷 (detect_stops)...")
stop_segments_initial = detect_stops(df)
print(f"偵測到停泊區段數量: {len(stop_segments_initial)}")


# ----------------------------
# 2. 傳統航程切分（供 QA / 地圖呈現）
# ----------------------------
print("\n[傳統] 切分航程（停泊→航行→停泊）...")
df_with_voyages, voyages_raw = split_voyages(df, stop_segments_initial)
print(f"切分後航程數量: {len(voyages_raw)}")

print("執行航程 QA 檢查...")
voyages_qc = voyage_quality_checker(
    df_with_voyages,
    voyages_raw,
    stop_segments=stop_segments_initial
)
print(f"QA 完成，有效航程數量: {voyages_qc['valid_flag'].sum()}")

print("輸出傳統航程地圖 voyages_map.html ...")
visualize_voyages(df_with_voyages, voyages_raw)
webbrowser.open("voyages_map.html")


# ============================
# 下方開始：新版「港到港＋錨泊/等待」流程
# ============================

# ----------------------------
# ★ Robust SOG stats：把最外側少量尖峰降噪後再算 std
# ----------------------------
def robust_sog_stats(sog_series: pd.Series, clip_pct: float = 2.5):
    """
    clip_pct=2.5 代表用中間 95% 做穩健估計（winsorize / trim）
    回傳：
      avg_sog
      sog_std            : 原始 std
      sog_p95            : 95分位（參考用）
      sog_std_w95        : winsorized std（建議拿來判 port_call）
      sog_std_trim95     : trimmed std（debug用）
    """
    x = pd.to_numeric(sog_series, errors="coerce").dropna().to_numpy()
    if len(x) == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    avg = float(np.mean(x))
    std = float(np.std(x, ddof=1)) if len(x) > 1 else 0.0
    p95 = float(np.percentile(x, 95))

    lo = float(np.percentile(x, clip_pct))
    hi = float(np.percentile(x, 100 - clip_pct))

    x_w = np.clip(x, lo, hi)
    std_w95 = float(np.std(x_w, ddof=1)) if len(x_w) > 1 else 0.0

    x_t = x[(x >= lo) & (x <= hi)]
    std_trim95 = float(np.std(x_t, ddof=1)) if len(x_t) > 1 else 0.0

    return avg, std, p95, std_w95, std_trim95


# ----------------------------
# 3. 從 stop_segments_initial 建立停泊 episodes + 合併碎段
# ----------------------------
def build_raw_stop_episodes(df, stop_segments):
    """
    將 detect_stops 回傳的 stop_segments(list of range)
    轉成一個 DataFrame：每列是一個「原始停泊 episode」
    新增 robust 指標：sog_std_w95 / sog_std_trim95
    """
    records = []
    for seg_id, seg in enumerate(stop_segments, start=1):
        idxs = list(seg)
        if not idxs:
            continue

        start_idx = idxs[0]
        end_idx   = idxs[-1]
        t_start = df.loc[start_idx, "Timestamp"]
        t_end   = df.loc[end_idx, "Timestamp"]
        duration_sec = (t_end - t_start).total_seconds()

        lat_center = df.loc[idxs, "Lat"].median()
        lon_center = df.loc[idxs, "Long"].median()

        avg_sog, sog_std, sog_p95, sog_std_w95, sog_std_trim95 = robust_sog_stats(df.loc[idxs, "Sog"], clip_pct=2.5)

        distances = [
            _safe_haversine(
                (lat_center, lon_center),
                (df.loc[j, "Lat"], df.loc[j, "Long"])
            )
            for j in idxs
        ]
        radius_km = float(np.percentile(distances, 95)) if distances else 0.0

        records.append({
            "raw_stop_id": seg_id,
            "start_idx": start_idx,
            "end_idx": end_idx,
            "t_start": t_start,
            "t_end": t_end,
            "duration_sec": duration_sec,
            "center_lat": lat_center,
            "center_lon": lon_center,
            "avg_sog": avg_sog,
            "sog_std": sog_std,
            "sog_p95": sog_p95,
            "sog_std_w95": sog_std_w95,        # ★ 用這個來判 port_call
            "sog_std_trim95": sog_std_trim95,  # debug
            "radius_km": radius_km,
        })

    return pd.DataFrame(records)


def merge_stop_episodes(df,
                        stops_raw,
                        max_gap_sec=30*60,
                        max_center_dist_km=1.0,
                        max_move_duration_sec=30*60,
                        max_move_disp_km=2.0,
                        max_move_avg_sog_kn=2.0):
    """
    合併相鄰停泊 episodes，合併後會重新計算 robust 指標
    """
    if stops_raw.empty:
        out = stops_raw.copy()
        for c in ["sog_p95", "sog_std_w95", "sog_std_trim95"]:
            if c not in out.columns:
                out[c] = np.nan
        return out.assign(stop_id=lambda x: np.arange(1, len(x)+1))

    stops_raw = stops_raw.sort_values("t_start").reset_index(drop=True)

    merged_records = []
    cur_start_idx = int(stops_raw.loc[0, "start_idx"])
    cur_end_idx   = int(stops_raw.loc[0, "end_idx"])

    for i in range(len(stops_raw) - 1):
        this_row = stops_raw.loc[i]
        next_row = stops_raw.loc[i+1]

        this_end_idx = int(this_row["end_idx"])
        next_start_idx = int(next_row["start_idx"])

        gap_sec = (next_row["t_start"] - this_row["t_end"]).total_seconds()

        center_dist_km = _safe_haversine(
            (this_row["center_lat"], this_row["center_lon"]),
            (next_row["center_lat"], next_row["center_lon"])
        )

        move_start_idx = this_end_idx + 1
        move_end_idx   = next_start_idx - 1

        has_move = move_start_idx <= move_end_idx
        move_duration_sec = 0.0
        move_disp_km = 0.0
        move_avg_sog = 0.0

        if has_move:
            t_move_start = df.loc[move_start_idx, "Timestamp"]
            t_move_end   = df.loc[move_end_idx, "Timestamp"]
            move_duration_sec = (t_move_end - t_move_start).total_seconds()
            move_disp_km = _safe_haversine(
                (df.loc[move_start_idx, "Lat"], df.loc[move_start_idx, "Long"]),
                (df.loc[move_end_idx,   "Lat"], df.loc[move_end_idx,   "Long"])
            )
            move_avg_sog = float(pd.to_numeric(df.loc[move_start_idx:move_end_idx, "Sog"], errors="coerce").mean())

        cond_time_close   = gap_sec <= max_gap_sec
        cond_center_close = center_dist_km <= max_center_dist_km

        if has_move:
            cond_move_small = (
                (move_duration_sec <= max_move_duration_sec) and
                (move_disp_km <= max_move_disp_km) and
                (move_avg_sog <= max_move_avg_sog_kn)
            )
        else:
            cond_move_small = True

        if cond_time_close and cond_center_close and cond_move_small:
            cur_end_idx = int(next_row["end_idx"])
        else:
            merged_records.append((cur_start_idx, cur_end_idx))
            cur_start_idx = int(next_row["start_idx"])
            cur_end_idx   = int(next_row["end_idx"])

    merged_records.append((cur_start_idx, cur_end_idx))

    merged_rows = []
    for stop_id, (s_idx, e_idx) in enumerate(merged_records, start=1):
        idxs = list(range(s_idx, e_idx + 1))
        t_start = df.loc[s_idx, "Timestamp"]
        t_end   = df.loc[e_idx, "Timestamp"]
        duration_sec = (t_end - t_start).total_seconds()

        lat_center = df.loc[idxs, "Lat"].median()
        lon_center = df.loc[idxs, "Long"].median()

        avg_sog, sog_std, sog_p95, sog_std_w95, sog_std_trim95 = robust_sog_stats(df.loc[idxs, "Sog"], clip_pct=2.5)

        distances = [
            _safe_haversine(
                (lat_center, lon_center),
                (df.loc[j, "Lat"], df.loc[j, "Long"])
            )
            for j in idxs
        ]
        radius_km = float(np.percentile(distances, 95)) if distances else 0.0

        merged_rows.append({
            "stop_id": stop_id,
            "start_idx": s_idx,
            "end_idx": e_idx,
            "t_start": t_start,
            "t_end": t_end,
            "duration_sec": duration_sec,
            "center_lat": lat_center,
            "center_lon": lon_center,
            "avg_sog": avg_sog,
            "sog_std": sog_std,
            "sog_p95": sog_p95,
            "sog_std_w95": sog_std_w95,        # ★ 用這個來判 port_call
            "sog_std_trim95": sog_std_trim95,  # debug
            "radius_km": radius_km
        })

    return pd.DataFrame(merged_rows)


print("\n建構停泊 episodes（raw）...")
stops_raw = build_raw_stop_episodes(df, stop_segments_initial)
print("原始停泊段數量:", len(stops_raw))

print("合併碎停泊 episodes（stops_merged）...")
stops_merged = merge_stop_episodes(df, stops_raw)
print("合併後停泊段數量:", len(stops_merged))


# ----------------------------
# 4. 載入港口清單 + 最近港口貼標 + 停泊型態分類
# ----------------------------
print("\n載入港口清單 filtered_ports.csv ...")
ports_df = pd.read_csv(ports_csv)

ports_df = ports_df.rename(columns={"unlocode": "port_id"})
if "port_name" not in ports_df.columns:
    ports_df["port_name"] = ports_df["port_id"]

print(f"港口數量: {len(ports_df)}")


def attach_nearest_port_to_stops(stops_df, ports_df):
    stops_df = stops_df.copy()

    port_coords = ports_df[["port_id", "port_name", "lat", "lon"]].to_dict(orient="records")

    nearest_ids = []
    nearest_names = []
    nearest_dists = []

    for _, row in stops_df.iterrows():
        lat0 = row["center_lat"]
        lon0 = row["center_lon"]

        best_port_id = None
        best_port_name = None
        best_dist = np.inf

        for p in port_coords:
            d = haversine((lat0, lon0), (p["lat"], p["lon"]))  # km
            if d < best_dist:
                best_dist = d
                best_port_id = p["port_id"]
                best_port_name = p["port_name"]

        nearest_ids.append(best_port_id)
        nearest_names.append(best_port_name)
        nearest_dists.append(best_dist)

    stops_df["nearest_port_id"] = nearest_ids
    stops_df["nearest_port_name"] = nearest_names
    stops_df["dist_to_port_km"] = nearest_dists

    return stops_df


def classify_stop_type(stops_df,
                       call_max_dist_km=15.0,
                       call_min_duration_sec=4*3600,
                       call_max_avg_sog_kn=0.1,
                       call_max_sog_std_w95_kn=0.2,   # ★改成 robust std
                       call_max_radius_km=0.5,
                       anch_max_dist_km=30.0,
                       anch_min_duration_sec=1800,
                       anch_min_avg_sog_kn=0.1,
                       anch_max_avg_sog_kn=2.0):
    """
    port_call 判斷改用 sog_std_w95（winsorized 95% std）：
      - 對少數尖峰更不敏感
      - 保留你原本想用 std 代表「穩定程度」的物理意義
    """
    stops_df = stops_df.copy()
    stops_df["stop_type"] = "other"

    if "sog_std_w95" not in stops_df.columns:
        stops_df["sog_std_w95"] = np.inf

    mask_call = (
        (stops_df["dist_to_port_km"] <= call_max_dist_km) &
        (stops_df["duration_sec"]   >= call_min_duration_sec) &
        (stops_df["avg_sog"].fillna(0) <= call_max_avg_sog_kn) &
        (stops_df["sog_std_w95"].fillna(np.inf) <= call_max_sog_std_w95_kn) &
        (stops_df["radius_km"].fillna(0) <= call_max_radius_km)
    )
    stops_df.loc[mask_call, "stop_type"] = "port_call"

    mask_anch = (
        (stops_df["stop_type"] == "other") &
        (stops_df["dist_to_port_km"] <= anch_max_dist_km) &
        (stops_df["duration_sec"]   >= anch_min_duration_sec) &
        #(stops_df["avg_sog"].fillna(0) >= anch_min_avg_sog_kn) &
        (stops_df["avg_sog"].fillna(0) <= anch_max_avg_sog_kn)
    )
    stops_df.loc[mask_anch, "stop_type"] = "anchorage_like"

    return stops_df


print("停泊 episodes 貼最近港口 & 分類型態...")
stops_with_port = attach_nearest_port_to_stops(stops_merged, ports_df)
stops_classified = classify_stop_type(stops_with_port)

print(stops_classified["stop_type"].value_counts())
display(stops_classified.head())


# ----------------------------
# 5. 從 port_call episodes 建立港到港 voyages + waiting_for_dest 標記
# ----------------------------
def build_port_calls(stops_classified):
    calls = stops_classified[stops_classified["stop_type"] == "port_call"].copy()
    if calls.empty:
        return pd.DataFrame(columns=[
            "call_id", "mmsi", "port_id", "port_name",
            "t_arrive_port", "t_depart_port", "stop_id"
        ])

    calls = calls.sort_values("t_start").reset_index(drop=True)
    calls["call_id"] = np.arange(1, len(calls) + 1)

    port_calls = pd.DataFrame({
        "call_id": calls["call_id"],
        "port_id": calls["nearest_port_id"],
        "port_name": calls["nearest_port_name"],
        "t_arrive_port": calls["t_start"],
        "t_depart_port": calls["t_end"],
        "stop_id": calls["stop_id"]
    })
    return port_calls


def build_voyages_from_port_calls(port_calls):
    if port_calls.empty or len(port_calls) < 2:
        return pd.DataFrame(columns=[
            "voyage_id", "origin_call_id", "dest_call_id",
            "origin_port_id", "dest_port_id",
            "origin_port_name", "dest_port_name",
            "t_depart_origin", "t_arrive_dest"
        ])

    port_calls = port_calls.sort_values("t_arrive_port").reset_index(drop=True)

    records = []
    voyage_id = 1
    for i in range(len(port_calls) - 1):
        row_o = port_calls.loc[i]
        row_d = port_calls.loc[i+1]

        records.append({
            "voyage_id": voyage_id,
            "origin_call_id": row_o["call_id"],
            "dest_call_id": row_d["call_id"],
            "origin_port_id": row_o["port_id"],
            "dest_port_id": row_d["port_id"],
            "origin_port_name": row_o["port_name"],
            "dest_port_name": row_d["port_name"],
            "t_depart_origin": row_o["t_depart_port"],
            "t_arrive_dest": row_d["t_arrive_port"]
        })
        voyage_id += 1

    return pd.DataFrame(records)


def label_waiting_for_dest(stops_classified, voyages_p2p,
                           ports_df,
                           alpha_progress=0.7,
                           max_dist_to_dest_km=100.0):
    stops = stops_classified.copy()
    stops["waiting_for_dest_port_id"] = None

    if voyages_p2p.empty:
        voyages_p2p["total_waiting_sec"] = 0.0
        return stops, voyages_p2p

    port_coord_map = {row["port_id"]: (row["lat"], row["lon"])
                      for _, row in ports_df[["port_id", "lat", "lon"]].iterrows()}

    voyages_records = []

    for _, v in voyages_p2p.iterrows():
        dest_port = v["dest_port_id"]
        t_dep = v["t_depart_origin"]
        t_arr = v["t_arrive_dest"]

        if pd.isna(t_dep) or pd.isna(t_arr) or (t_arr <= t_dep):
            v_dict = v.to_dict()
            v_dict["total_waiting_sec"] = 0.0
            voyages_records.append(v_dict)
            continue

        T_voy = (t_arr - t_dep).total_seconds()
        if T_voy <= 0:
            v_dict = v.to_dict()
            v_dict["total_waiting_sec"] = 0.0
            voyages_records.append(v_dict)
            continue

        mask_in_voy = (stops["t_start"] >= t_dep) & (stops["t_start"] <= t_arr)
        stops_voy = stops[mask_in_voy].copy()

        dest_lat, dest_lon = port_coord_map.get(dest_port, (None, None))

        waiting_ids = []
        total_waiting_sec = 0.0

        for _, s in stops_voy.iterrows():
            if s["stop_type"] != "anchorage_like":
                continue

            t_mid = s["t_start"] + (s["t_end"] - s["t_start"]) / 2
            progress = (t_mid - t_dep).total_seconds() / T_voy
            if progress < alpha_progress:
                continue

            dist_to_B = np.inf
            if dest_lat is not None:
                dist_to_B = haversine((s["center_lat"], s["center_lon"]), (dest_lat, dest_lon))
            if dist_to_B > max_dist_to_dest_km:
                continue

            waiting_ids.append(s["stop_id"])
            total_waiting_sec += s["duration_sec"]

        if waiting_ids:
            stops.loc[stops["stop_id"].isin(waiting_ids), "waiting_for_dest_port_id"] = dest_port

        v_dict = v.to_dict()
        v_dict["total_waiting_sec"] = total_waiting_sec
        voyages_records.append(v_dict)

    voyages_with_waiting = pd.DataFrame(voyages_records)
    return stops, voyages_with_waiting
# ----------------------------
# 6. 標記有large_time_gap 的航程
# ----------------------------
def mark_large_time_gap_for_voyages(
    df_points: pd.DataFrame,
    voyages_df: pd.DataFrame,
    time_col: str = "Timestamp",
    lat_col: str = "Lat",
    lon_col: str = "Long",
    dep_col: str = "t_depart_origin",
    arr_col: str = "t_arrive_dest",
    voyage_id_col: str = "voyage_id",
    gap_sec_th: float = 6 * 3600,   # 例如 6 小時；你可改 3hr/12hr/24hr
    window_pad_sec: float = 0.0,    # 需要可加 padding
    verbose: bool = False
):
    """
    對每條 voyage（用 dep~arr 時間窗切點），計算最大 dt_sec，並標記 large_time_gap。
    回傳：voyages_df (新增欄位)
    """
    d = df_points.sort_values(time_col).copy()
    v = voyages_df.copy()

    # 初始化欄位
    v["n_points"] = 0
    v["max_dt_sec"] = np.nan
    v["max_dt_hr"] = np.nan
    v["gap_from"] = pd.NaT
    v["gap_to"] = pd.NaT
    v["large_time_gap_flag"] = False

    for i, row in v.iterrows():
        vid = row[voyage_id_col]
        t0 = row[dep_col]
        t1 = row[arr_col]
        if pd.isna(t0) or pd.isna(t1) or t1 <= t0:
            continue

        t0p = t0 - pd.Timedelta(seconds=window_pad_sec)
        t1p = t1 + pd.Timedelta(seconds=window_pad_sec)

        seg = d[(d[time_col] >= t0p) & (d[time_col] <= t1p)].copy()
        if len(seg) < 2:
            v.at[i, "n_points"] = len(seg)
            continue

        seg["dt_sec"] = seg[time_col].diff().dt.total_seconds()

        # 最大 gap（忽略第一筆 NaN）
        jmax = seg["dt_sec"].idxmax()
        max_dt = float(seg.loc[jmax, "dt_sec"])

        v.at[i, "n_points"] = len(seg)
        v.at[i, "max_dt_sec"] = max_dt
        v.at[i, "max_dt_hr"] = max_dt / 3600.0 if np.isfinite(max_dt) else np.nan

        if np.isfinite(max_dt):
            # gap_from 是上一筆時間（idxmax 那列的前一列）
            pos = seg.index.get_loc(jmax)
            if pos > 0:
                prev_idx = seg.index[pos - 1]
                v.at[i, "gap_from"] = seg.loc[prev_idx, time_col]
                v.at[i, "gap_to"] = seg.loc[jmax, time_col]

            v.at[i, "large_time_gap_flag"] = (max_dt >= gap_sec_th)

        if verbose and v.at[i, "large_time_gap_flag"]:
            print(f"[voyage {vid}] max_dt_hr={v.at[i,'max_dt_hr']:.2f}  gap_from={v.at[i,'gap_from']}  gap_to={v.at[i,'gap_to']}")

    return v


print("\n從停泊 episodes 建立 port_calls & 港到港 voyages...")
port_calls = build_port_calls(stops_classified)
voyages_p2p = build_voyages_from_port_calls(port_calls)

print("Port calls 數量:", len(port_calls))
print("港到港 voyages 數量:", len(voyages_p2p))

print("標記 waiting_for_dest_port_id（錨泊/等待段）...")
stops_labeled, voyages_with_waiting = label_waiting_for_dest(
    stops_classified,
    voyages_p2p,
    ports_df,
    alpha_progress=0.7,
    max_dist_to_dest_km=100.0
)

port_calls["mmsi"] = target_mmsi
voyages_with_waiting["mmsi"] = target_mmsi

print("有等待時間的港到港航程數量:",
      (voyages_with_waiting["total_waiting_sec"] > 0).sum())



print("\n=== stops_labeled 範例 ===")
display(stops_labeled.head())

print("\n=== voyages_with_waiting 範例 ===")
display(voyages_with_waiting.head())

#print("\n流程完成，可以開始拿 voyages_with_waiting / stops_labeled 來準備 ETA 的訓練資料了。")

voyages_with_waiting = mark_large_time_gap_for_voyages(
    df_points=df,  # 你的 AIS 點資料（teleport 清過的 df）
    voyages_df=voyages_with_waiting,
    gap_sec_th=6*3600,
    verbose=True
)

display(voyages_with_waiting.sort_values("max_dt_sec", ascending=False).head(10))
print("large_time_gap 航程數量:", voyages_with_waiting["large_time_gap_flag"].sum())

# =========================
# Loitering / Slow-wait anomaly detector (non-stop) - drop-in cell
# Assumes you already ran the pipeline above and have:
#   df (teleport-cleaned points, with Timestamp/Lat/Long/Sog)
#   voyages_with_waiting (port-to-port voyages with t_depart_origin/t_arrive_dest, voyage_id)
# =========================

import numpy as np
import pandas as pd
from math import radians, sin, cos, atan2, sqrt

# ---------- utils ----------
def _to_lon180(x: float) -> float:
    return ((x + 180.0) % 360.0) - 180.0

def haversine_km(lat1, lon1, lat2, lon2) -> float:
    R = 6371.0088
    p1, p2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlambda = radians(_to_lon180(lon2 - lon1))
    a = sin(dphi/2)**2 + cos(p1)*cos(p2)*sin(dlambda/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1-a))

def _path_len_km(lat, lon):
    if len(lat) < 2:
        return 0.0
    tot = 0.0
    for i in range(1, len(lat)):
        if np.isnan(lat[i-1]) or np.isnan(lon[i-1]) or np.isnan(lat[i]) or np.isnan(lon[i]):
            continue
        tot += haversine_km(lat[i-1], lon[i-1], lat[i], lon[i])
    return float(tot)

def _r95_km(lat, lon):
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    mask = ~np.isnan(lat) & ~np.isnan(lon)
    if mask.sum() == 0:
        return np.nan
    latm = float(np.nanmedian(lat[mask]))
    lonm = float(np.nanmedian(lon[mask]))
    d = np.array([haversine_km(latm, lonm, a, b) for a, b in zip(lat[mask], lon[mask])], dtype=float)
    return float(np.nanpercentile(d, 95)) if len(d) else np.nan

def _net_disp_km(lat, lon):
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    mask = ~np.isnan(lat) & ~np.isnan(lon)
    if mask.sum() < 2:
        return 0.0
    i0 = np.where(mask)[0][0]
    i1 = np.where(mask)[0][-1]
    return float(haversine_km(lat[i0], lon[i0], lat[i1], lon[i1]))

def _winsorize_std(x, clip_pct=2.5):
    x = pd.to_numeric(pd.Series(x), errors="coerce").dropna().to_numpy(dtype=float)
    if len(x) == 0:
        return np.nan
    if len(x) == 1:
        return 0.0
    lo = float(np.percentile(x, clip_pct))
    hi = float(np.percentile(x, 100 - clip_pct))
    xw = np.clip(x, lo, hi)
    return float(np.std(xw, ddof=1))

def _merge_intervals(intervals, max_gap_sec=600):
    """
    intervals: list of (start_ts, end_ts)
    """
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda x: x[0])
    out = [list(intervals[0])]
    for s, e in intervals[1:]:
        if (s - out[-1][1]).total_seconds() <= max_gap_sec:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return [(a, b) for a, b in out]

# ---------- main detector ----------
def detect_loitering_events_for_voyages(
    df_points: pd.DataFrame,
    voyages_df: pd.DataFrame,
    time_col: str = "Timestamp",
    lat_col: str = "Lat",
    lon_col: str = "Long",
    sog_col: str = "Sog",
    dep_col: str = "t_depart_origin",
    arr_col: str = "t_arrive_dest",
    voyage_id_col: str = "voyage_id",

    # windowing
    window_sec: int = 10 * 60,        # 10 min window
    step_sec: int = 2 * 60,           # slide every 2 min
    min_points_in_window: int = 12,   # depends on your sampling; 12 points / 10min = ~50s interval

    # anomaly logic (tune later)
    sog_low_kn: float = 0.5,
    sog_high_kn: float = 6.0,
    low_speed_ratio_th: float = 0.70,

    r95_max_km: float = 3.0,          # "small area" but not necessarily stopped
    net_disp_max_km: float = 1.0,     # little net progress within window
    tortuosity_min: float = 2.5,      # path / net_disp (higher => circling)
    sog_std_w95_min: float = 0.20,    # "moving around" signal (not dead-still)

    min_event_sec: int = 20 * 60,     # event must last >= 20 min
    merge_gap_sec: int = 10 * 60,     # merge nearby candidate windows
    verbose: bool = True
):
    d = df_points.sort_values(time_col).copy()
    # ensure datetime
    d[time_col] = pd.to_datetime(d[time_col])

    voyages = voyages_df.copy()
    voyages[dep_col] = pd.to_datetime(voyages[dep_col])
    voyages[arr_col] = pd.to_datetime(voyages[arr_col])

    all_events = []
    point_labels = []  # store (voyage_id, start_ts, end_ts, event_id)

    event_id = 1

    for _, v in voyages.iterrows():
        vid = v[voyage_id_col]
        t0 = v[dep_col]
        t1 = v[arr_col]
        if pd.isna(t0) or pd.isna(t1) or t1 <= t0:
            continue

        seg = d[(d[time_col] >= t0) & (d[time_col] <= t1)].copy()
        if len(seg) < min_points_in_window:
            continue

        # build sliding windows
        t_start = seg[time_col].iloc[0]
        t_end = seg[time_col].iloc[-1]
        cur = t_start

        candidate_intervals = []

        while cur <= t_end:
            w_end = cur + pd.Timedelta(seconds=window_sec)
            w = seg[(seg[time_col] >= cur) & (seg[time_col] < w_end)]
            if len(w) >= min_points_in_window:
                lat = w[lat_col].astype(float).to_numpy()
                lon = w[lon_col].astype(float).to_numpy()
                sog = pd.to_numeric(w[sog_col], errors="coerce").to_numpy(dtype=float)

                # speed band ratio
                band = (sog >= sog_low_kn) & (sog <= sog_high_kn)
                low_ratio = float(np.nanmean(band)) if len(sog) else 0.0
                sog_std_w95 = _winsorize_std(sog, clip_pct=2.5)

                # geometry
                r95 = _r95_km(lat, lon)
                net = _net_disp_km(lat, lon)
                plen = _path_len_km(lat, lon)
                tort = float(plen / max(net, 1e-6))

                # candidate rule: (low speed) + (small area) + (either circling OR jittery movement)
                cond_speed = (low_ratio >= low_speed_ratio_th)
                cond_area = (np.isfinite(r95) and r95 <= r95_max_km and net <= net_disp_max_km)

                # "loitering-ness": circling or back-and-forth
                cond_circling = (tort >= tortuosity_min)
                cond_jitter = (np.isfinite(sog_std_w95) and sog_std_w95 >= sog_std_w95_min) and (net <= net_disp_max_km)

                if cond_speed and cond_area and (cond_circling or cond_jitter):
                    candidate_intervals.append((cur, min(w_end, t_end)))

            cur = cur + pd.Timedelta(seconds=step_sec)

        # merge candidates -> events
        merged = _merge_intervals(candidate_intervals, max_gap_sec=merge_gap_sec)
        # filter by duration
        merged = [(a, b) for (a, b) in merged if (b - a).total_seconds() >= min_event_sec]

        if verbose and merged:
            print(f"[loitering] voyage {vid}: found {len(merged)} events")

        for (a, b) in merged:
            all_events.append({
                "event_id": event_id,
                "voyage_id": vid,
                "t_start": a,
                "t_end": b,
                "duration_sec": float((b - a).total_seconds()),
                "anomaly_type": "loitering_or_slow_wait"
            })
            point_labels.append((vid, a, b, event_id))
            event_id += 1

    loitering_events = pd.DataFrame(all_events)

    # point-level labeling (vector-ish per voyage)
    df_lab = d.copy()
    df_lab["loitering_flag"] = False
    df_lab["loitering_event_id"] = np.nan

    if not loitering_events.empty:
        # assign by scanning labels (fast enough for per-voyage volumes)
        for (vid, a, b, eid) in point_labels:
            m = (df_lab[time_col] >= a) & (df_lab[time_col] <= b)
            # We don't have voyage_id on points; we label by time only (safe if single MMSI per df).
            # If you later run multi-MMSI, add an MMSI column and filter it here.
            df_lab.loc[m, "loitering_flag"] = True
            # if overlap, keep smallest id (or you can keep the latest)
            cur_e = df_lab.loc[m, "loitering_event_id"]
            df_lab.loc[m, "loitering_event_id"] = np.where(cur_e.isna(), eid, np.minimum(cur_e, eid))

    return loitering_events, df_lab


# ---------- run ----------
loitering_events, df_loitering_labeled = detect_loitering_events_for_voyages(
    df_points=df,
    voyages_df=voyages_with_waiting,
    window_sec=10*60,
    step_sec=2*60,
    min_points_in_window=4,

    sog_low_kn=0.5,
    sog_high_kn=6.0,
    low_speed_ratio_th=0.70,

    r95_max_km=5.0,
    net_disp_max_km=2.0,
    tortuosity_min=2.5,
    sog_std_w95_min=0.20,

    min_event_sec=10*60,
    merge_gap_sec=10*60,
    verbose=True
)

display(loitering_events.head(10))
print("Total loitering events:", len(loitering_events))

# ---------- voyage-level aggregation ----------
voyages_with_waiting_enhanced = voyages_with_waiting.copy()
voyages_with_waiting_enhanced["total_loitering_sec"] = 0.0
voyages_with_waiting_enhanced["loitering_events_count"] = 0

if not loitering_events.empty:
    agg = loitering_events.groupby("voyage_id").agg(
        total_loitering_sec=("duration_sec", "sum"),
        loitering_events_count=("event_id", "count")
    ).reset_index()
    voyages_with_waiting_enhanced = voyages_with_waiting_enhanced.merge(
        agg, on="voyage_id", how="left", suffixes=("", "_new")
    )
    voyages_with_waiting_enhanced["total_loitering_sec"] = voyages_with_waiting_enhanced["total_loitering_sec_new"].fillna(0.0)
    voyages_with_waiting_enhanced["loitering_events_count"] = voyages_with_waiting_enhanced["loitering_events_count_new"].fillna(0).astype(int)
    voyages_with_waiting_enhanced = voyages_with_waiting_enhanced.drop(
        columns=[c for c in ["total_loitering_sec_new", "loitering_events_count_new"] if c in voyages_with_waiting_enhanced.columns]
    )

display(voyages_with_waiting_enhanced.sort_values("total_loitering_sec", ascending=False).head(10))

# (optional) save
# out_dir = Path("outputs")
# out_dir.mkdir(exist_ok=True)
# loitering_events.to_csv(out_dir/"loitering_events.csv", index=False, encoding="utf-8-sig")
# df_loitering_labeled.to_parquet(out_dir/"df_loitering_labeled.parquet", index=False)
# voyages_with_waiting_enhanced.to_csv(out_dir/"voyages_with_waiting_enhanced.csv", index=False, encoding="utf-8-sig")

import numpy as np
import pandas as pd

def _summarize_loitering_to_voyages_no_suffix(
    voyages_df: pd.DataFrame,
    events_df: pd.DataFrame,
    *,
    voyage_id_col="voyage_id",
    event_start_col="t_start",
    event_end_col="t_end",
    out_total_col="total_loitering_sec",
    out_cnt_col="loitering_events_count",
):
    vw = voyages_df.copy()

    # 避免 merge 產生 _x/_y
    for c in [out_total_col, out_cnt_col]:
        if c in vw.columns:
            vw = vw.drop(columns=c)

    # 沒事件就補 0
    if events_df is None or (isinstance(events_df, pd.DataFrame) and events_df.empty):
        vw[out_total_col] = 0.0
        vw[out_cnt_col] = 0
        return vw

    ev = events_df.copy()
    ev[voyage_id_col] = pd.to_numeric(ev[voyage_id_col], errors="coerce").astype("Int64")
    ev[event_start_col] = pd.to_datetime(ev[event_start_col], errors="coerce")
    ev[event_end_col]   = pd.to_datetime(ev[event_end_col], errors="coerce")
    ev = ev.dropna(subset=[voyage_id_col, event_start_col, event_end_col]).copy()

    if ev.empty:
        vw[out_total_col] = 0.0
        vw[out_cnt_col] = 0
        return vw

    if "duration_sec" not in ev.columns:
        ev["duration_sec"] = (ev[event_end_col] - ev[event_start_col]).dt.total_seconds().clip(lower=0)

    g = ev.groupby(voyage_id_col)["duration_sec"].agg(["sum", "count"]).reset_index()
    g = g.rename(columns={"sum": out_total_col, "count": out_cnt_col})

    vw[voyage_id_col] = pd.to_numeric(vw[voyage_id_col], errors="coerce").astype("Int64")
    vw = vw.merge(g, on=voyage_id_col, how="left")
    vw[out_total_col] = vw[out_total_col].fillna(0.0)
    vw[out_cnt_col]   = vw[out_cnt_col].fillna(0).astype(int)
    return vw


def edge_guard_loitering_events_by_first_last_k_points(
    df_points: pd.DataFrame,
    voyages_df: pd.DataFrame,
    events_df: pd.DataFrame,
    k: int = 3,
    time_col: str = "Timestamp",
    dep_col: str = "t_depart_origin",
    arr_col: str = "t_arrive_dest",
    voyage_id_col: str = "voyage_id",
    event_start_col: str = "t_start",
    event_end_col: str = "t_end",
):
    # --- 空事件 ---
    if events_df is None or not isinstance(events_df, pd.DataFrame) or events_df.empty:
        v_upd = voyages_df.copy()
        v_upd = _summarize_loitering_to_voyages_no_suffix(v_upd, pd.DataFrame(), voyage_id_col=voyage_id_col,
                                                         event_start_col=event_start_col, event_end_col=event_end_col)
        empty = events_df.copy() if isinstance(events_df, pd.DataFrame) else pd.DataFrame()
        return empty, empty.iloc[0:0].copy(), v_upd

    d = df_points.copy()
    d[time_col] = pd.to_datetime(d[time_col], errors="coerce")

    v = voyages_df.copy()
    v[dep_col] = pd.to_datetime(v[dep_col], errors="coerce")
    v[arr_col] = pd.to_datetime(v[arr_col], errors="coerce")
    v[voyage_id_col] = pd.to_numeric(v[voyage_id_col], errors="coerce").astype("Int64")

    e = events_df.copy()
    e[event_start_col] = pd.to_datetime(e[event_start_col], errors="coerce")
    e[event_end_col]   = pd.to_datetime(e[event_end_col], errors="coerce")
    e[voyage_id_col]   = pd.to_numeric(e[voyage_id_col], errors="coerce").astype("Int64")

    e = e.dropna(subset=[voyage_id_col, event_start_col, event_end_col]).copy()
    if e.empty:
        v_upd = _summarize_loitering_to_voyages_no_suffix(v, e, voyage_id_col=voyage_id_col,
                                                         event_start_col=event_start_col, event_end_col=event_end_col)
        return e, e.iloc[0:0].copy(), v_upd

    # 只處理 events 出現過的 voyage_id
    vids = e[voyage_id_col].dropna().astype(int).unique().tolist()

    # vid -> (head_min, head_max, tail_min, tail_max)
    edge_ranges = {}
    for vid in vids:
        rows = v[v[voyage_id_col] == vid]
        if rows.empty:
            continue
        row = rows.iloc[0]
        t0, t1 = row[dep_col], row[arr_col]
        if pd.isna(t0) or pd.isna(t1) or t1 <= t0:
            continue

        seg = d[(d[time_col] >= t0) & (d[time_col] <= t1)].dropna(subset=[time_col]).sort_values(time_col)
        if len(seg) == 0:
            continue

        head = seg.head(k)[time_col].to_list()
        tail = seg.tail(k)[time_col].to_list()
        edge_ranges[int(vid)] = (min(head), max(head), min(tail), max(tail))

    def _overlap(a0, a1, b0, b1):
        return (a0 <= b1) and (a1 >= b0)

    keep_mask = []
    for _, r in e.iterrows():
        vid = int(r[voyage_id_col])
        if vid not in edge_ranges:
            keep_mask.append(True)
            continue

        h0, h1, t0, t1 = edge_ranges[vid]
        es, ee = r[event_start_col], r[event_end_col]
        hit_head = _overlap(es, ee, h0, h1)
        hit_tail = _overlap(es, ee, t0, t1)
        keep_mask.append(not (hit_head or hit_tail))

    keep_mask = np.array(keep_mask, dtype=bool)
    events_kept = e.loc[keep_mask].copy().reset_index(drop=True)
    events_dropped = e.loc[~keep_mask].copy().reset_index(drop=True)

    # 回填 voyages（不產生 _x/_y）
    v_upd = _summarize_loitering_to_voyages_no_suffix(
        v, events_kept,
        voyage_id_col=voyage_id_col,
        event_start_col=event_start_col,
        event_end_col=event_end_col,
        out_total_col="total_loitering_sec",
        out_cnt_col="loitering_events_count",
    )

    return events_kept, events_dropped, v_upd


# === 套用：包含起/迄點前後 3 點 ===
voyages_df = voyages_with_waiting_enhanced.copy() if "voyages_with_waiting_enhanced" in globals() else voyages_with_waiting.copy()

#  建議：edge-guard 最好吃「raw events」（merge 前）
# 如果你真的要 guard merged 也可以，把 base_events 改成 loitering_events_merged
base_events = loitering_events

loitering_events_kept_3pts, loitering_events_dropped_3pts, voyages_with_waiting_edgeguarded_3pts = \
    edge_guard_loitering_events_by_first_last_k_points(
        df_points=df,
        voyages_df=voyages_df,
        events_df=base_events,
        k=3
    )

print("edge-guard(前後3點) before:", len(base_events))
print("edge-guard(前後3點) kept  :", len(loitering_events_kept_3pts))
print("edge-guard(前後3點) drop  :", len(loitering_events_dropped_3pts))

display(voyages_with_waiting_edgeguarded_3pts[["voyage_id","total_loitering_sec","loitering_events_count"]].head())

import numpy as np
import pandas as pd
from math import radians, sin, cos, atan2, sqrt

# -----------------------
# helpers
# -----------------------
def _to_lon180(dlon):
    return ((dlon + 180.0) % 360.0) - 180.0

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlmb = radians(_to_lon180(lon2 - lon1))
    a = sin(dphi/2)**2 + cos(p1)*cos(p2)*sin(dlmb/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1-a))

def segment_r95_km(seg: pd.DataFrame, lat_col="Lat", lon_col="Long"):
    if seg.empty:
        return np.nan
    lat = pd.to_numeric(seg[lat_col], errors="coerce").to_numpy()
    lon = pd.to_numeric(seg[lon_col], errors="coerce").to_numpy()
    lat_med = np.nanmedian(lat)
    lon_med = np.nanmedian(lon)
    d = np.array([haversine_km(lat_med, lon_med, a, b) for a, b in zip(lat, lon)], dtype=float)
    d = d[np.isfinite(d)]
    if d.size == 0:
        return np.nan
    return float(np.percentile(d, 95))

def _ensure_datetime(df, cols):
    df = df.copy()
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

# -----------------------
# core
# -----------------------
def merge_loitering_events_by_compactness(
    df_points: pd.DataFrame,
    voyages_df: pd.DataFrame,
    events_df: pd.DataFrame,
    *,
    time_col="Timestamp", lat_col="Lat", lon_col="Long",
    dep_col="t_depart_origin", arr_col="t_arrive_dest",
    voyage_id_col="voyage_id",
    event_id_col="event_id",
    ev_start_col="t_start", ev_end_col="t_end",
    merge_r95_max_km: float | None = None,
    max_event_hr: float | None = None,
    max_gap_hr: float = 24.0,
    min_points_in_span: int = 5,
):
    """
    將 loitering events（建議：已做完 3pt edge-guard 的 kept events）
    依「同一區域緊湊性」合併：
      - gap <= max_gap_hr
      - 合併後時間跨度 <= max_event_hr
      - 合併後 r95 <= merge_r95_max_km 且 span 點數 >= min_points_in_span
    並回填每個 voyage 的 total_loitering_sec / loitering_events_count（不要求 voyage 原本有欄位）
    """
    # points
    d = df_points.copy()
    d[time_col] = pd.to_datetime(d[time_col], errors="coerce")
    d = d.dropna(subset=[time_col]).sort_values(time_col)

    # voyages
    v = voyages_df.copy()
    v = _ensure_datetime(v, [dep_col, arr_col])
    if voyage_id_col not in v.columns:
        raise KeyError(f"voyages_df missing column: {voyage_id_col}")

    # events
    e = events_df.copy() if isinstance(events_df, pd.DataFrame) else pd.DataFrame()

    # empty -> return safe outputs
    if e.empty:
        merged = pd.DataFrame(columns=[
            "event_id", voyage_id_col, ev_start_col, ev_end_col, "duration_sec",
            "anomaly_type", "r95_km", "n_points", "merged_from_event_ids"
        ])
        v_out = v.copy()
        # 每次都覆蓋建立，避免 KeyError
        v_out["total_loitering_sec"] = 0.0
        v_out["loitering_events_count"] = 0
        return merged, v_out

    # ensure datetime
    e = _ensure_datetime(e, [ev_start_col, ev_end_col])

    # try to attach voyage_id if missing
    if voyage_id_col not in e.columns:
        e[voyage_id_col] = np.nan
        mids = pd.to_datetime(e[ev_start_col], errors="coerce") + (
            pd.to_datetime(e[ev_end_col], errors="coerce") - pd.to_datetime(e[ev_start_col], errors="coerce")
        ) / 2
        for idx, mid in zip(e.index, mids):
            if pd.isna(mid):
                continue
            hit = v[(v[dep_col] <= mid) & (v[arr_col] >= mid)]
            if not hit.empty:
                e.at[idx, voyage_id_col] = hit.iloc[0][voyage_id_col]

    # sanitize
    e = e.dropna(subset=[voyage_id_col, ev_start_col, ev_end_col]).copy()
    e[voyage_id_col] = pd.to_numeric(e[voyage_id_col], errors="coerce")
    e = e.dropna(subset=[voyage_id_col]).copy()
    e[voyage_id_col] = e[voyage_id_col].astype(int)

    # event_id fallback
    if event_id_col not in e.columns:
        e[event_id_col] = np.arange(1, len(e) + 1)

    # defaults
    if merge_r95_max_km is None:
        merge_r95_max_km = 10.0

    if max_event_hr is None:
        if "duration_sec" in e.columns and len(e) >= 5:
            dur_hr = pd.to_numeric(e["duration_sec"], errors="coerce") / 3600.0
            base = float(dur_hr.quantile(0.95)) if dur_hr.notna().any() else 6.0
            max_event_hr = float(np.clip(base * 3.0, 6.0, 48.0))
        else:
            max_event_hr = 24.0

    merged_rows = []

    # per voyage
    for vid, evs in e.sort_values([voyage_id_col, ev_start_col]).groupby(voyage_id_col):
        evs = evs.sort_values(ev_start_col).reset_index(drop=True)

        vrow = v[v[voyage_id_col] == vid]
        if vrow.empty:
            continue
        t0 = vrow.iloc[0][dep_col]
        t1 = vrow.iloc[0][arr_col]
        if pd.isna(t0) or pd.isna(t1) or t1 <= t0:
            continue

        seg_v = d[(d[time_col] >= t0) & (d[time_col] <= t1)].copy()
        seg_v = seg_v.sort_values(time_col)
        if seg_v.empty:
            continue

        cur_start = pd.to_datetime(evs.loc[0, ev_start_col])
        cur_end   = pd.to_datetime(evs.loc[0, ev_end_col])
        member_ids = [int(evs.loc[0, event_id_col])]

        def _span_r95(a, b):
            sub = seg_v[(seg_v[time_col] >= a) & (seg_v[time_col] <= b)]
            return segment_r95_km(sub, lat_col=lat_col, lon_col=lon_col), len(sub)

        for j in range(1, len(evs)):
            ns = pd.to_datetime(evs.loc[j, ev_start_col])
            ne = pd.to_datetime(evs.loc[j, ev_end_col])
            if pd.isna(ns) or pd.isna(ne):
                continue

            gap_hr = (ns - cur_end).total_seconds() / 3600.0

            # too far -> flush
            if gap_hr > max_gap_hr:
                merged_rows.append({
                    voyage_id_col: vid,
                    ev_start_col: cur_start,
                    ev_end_col: cur_end,
                    "duration_sec": (cur_end - cur_start).total_seconds(),
                    "merged_from_event_ids": member_ids,
                })
                cur_start, cur_end = ns, ne
                member_ids = [int(evs.loc[j, event_id_col])]
                continue

            cand_start, cand_end = cur_start, max(cur_end, ne)
            dur_hr = (cand_end - cand_start).total_seconds() / 3600.0

            if dur_hr > max_event_hr:
                merged_rows.append({
                    voyage_id_col: vid,
                    ev_start_col: cur_start,
                    ev_end_col: cur_end,
                    "duration_sec": (cur_end - cur_start).total_seconds(),
                    "merged_from_event_ids": member_ids,
                })
                cur_start, cur_end = ns, ne
                member_ids = [int(evs.loc[j, event_id_col])]
                continue

            r95, npts = _span_r95(cand_start, cand_end)

            if np.isfinite(r95) and (r95 <= merge_r95_max_km) and (npts >= min_points_in_span):
                cur_end = cand_end
                member_ids.append(int(evs.loc[j, event_id_col]))
            else:
                merged_rows.append({
                    voyage_id_col: vid,
                    ev_start_col: cur_start,
                    ev_end_col: cur_end,
                    "duration_sec": (cur_end - cur_start).total_seconds(),
                    "merged_from_event_ids": member_ids,
                })
                cur_start, cur_end = ns, ne
                member_ids = [int(evs.loc[j, event_id_col])]

        merged_rows.append({
            voyage_id_col: vid,
            ev_start_col: cur_start,
            ev_end_col: cur_end,
            "duration_sec": (cur_end - cur_start).total_seconds(),
            "merged_from_event_ids": member_ids,
        })

    merged = pd.DataFrame(merged_rows)

    # still empty -> safe return
    if merged.empty:
        merged = pd.DataFrame(columns=[
            "event_id", voyage_id_col, ev_start_col, ev_end_col, "duration_sec",
            "anomaly_type", "r95_km", "n_points", "merged_from_event_ids"
        ])
        v_out = v.copy()
        v_out["total_loitering_sec"] = 0.0
        v_out["loitering_events_count"] = 0
        return merged, v_out

    merged = merged.sort_values([voyage_id_col, ev_start_col]).reset_index(drop=True)
    merged["event_id"] = np.arange(1, len(merged) + 1)
    merged["anomaly_type"] = "loitering_merged"

    # compute merged r95 / n_points
    r95_list, npts_list = [], []
    for _, row in merged.iterrows():
        vid = int(row[voyage_id_col])
        vrow = v[v[voyage_id_col] == vid]
        if vrow.empty:
            r95_list.append(np.nan); npts_list.append(0); continue
        t0 = vrow.iloc[0][dep_col]
        t1 = vrow.iloc[0][arr_col]
        seg_v = d[(d[time_col] >= t0) & (d[time_col] <= t1)].copy()
        sub = seg_v[(seg_v[time_col] >= row[ev_start_col]) & (seg_v[time_col] <= row[ev_end_col])]
        npts_list.append(int(len(sub)))
        r95_list.append(segment_r95_km(sub, lat_col=lat_col, lon_col=lon_col))

    merged["n_points"] = npts_list
    merged["r95_km"] = r95_list

    # --- summarize back to voyages (safe, no KeyError) ---
    v_out = v.copy()

    # 永遠先清掉舊 summary 欄，避免 _x/_y 與 KeyError
    v_out = v_out.drop(columns=[
        "total_loitering_sec", "loitering_events_count",
        "total_loitering_sec_x", "total_loitering_sec_y",
        "loitering_events_count_x", "loitering_events_count_y",
    ], errors="ignore")

    agg = merged.groupby(voyage_id_col)["duration_sec"].agg(["sum", "count"]).reset_index()
    agg = agg.rename(columns={"sum": "total_loitering_sec", "count": "loitering_events_count"})

    v_out = v_out.merge(agg, on=voyage_id_col, how="left")
    v_out["total_loitering_sec"] = v_out["total_loitering_sec"].fillna(0.0)
    v_out["loitering_events_count"] = v_out["loitering_events_count"].fillna(0).astype(int)

    return merged, v_out


# -----------------------
# RUN (改成吃 3pt edge-guard 後的 events)
# -----------------------
df_points = df
vw = voyages_with_waiting_enhanced.copy() if "voyages_with_waiting_enhanced" in globals() else voyages_with_waiting.copy()

#  正確：merge 前先 edge-guard，所以這裡優先用 3pt kept events
if "loitering_events_kept_3pts" in globals():
    base_events_for_merge = loitering_events_kept_3pts
elif "loitering_events_kept" in globals():
    base_events_for_merge = loitering_events_kept
else:
    # 退而求其次：你手上現成的（但建議你確定它是 edge-guard 後的）
    base_events_for_merge = loitering_events

loitering_events_merged, voyages_with_waiting_merged = merge_loitering_events_by_compactness(
    df_points=df_points,
    voyages_df=vw,
    events_df=base_events_for_merge,
    merge_r95_max_km=10.0,
    max_event_hr=24.0,
    max_gap_hr=24.0,
)

display(loitering_events_merged.head(10))
display(voyages_with_waiting_merged[["voyage_id","total_loitering_sec","loitering_events_count"]].head(10))






載入與前處理資料中...
  已對 MMSI=477300400 進行 SOG 校正（/10）
清理後資料筆數: 200878
移除瞬移(teleport)雜訊點（smart迭代：先掃描、有異常才跑下一輪）...


C:\Users\slab\AppData\Local\Temp\ipykernel_13168\817378119.py:171: RuntimeWarning: divide by zero encountered in divide
  speed_kn = (dist / (dt.to_numpy() / 3600.0)) / 1.852


[teleport] iter 1: dropped 334 points


C:\Users\slab\AppData\Local\Temp\ipykernel_13168\817378119.py:171: RuntimeWarning: divide by zero encountered in divide
  speed_kn = (dist / (dt.to_numpy() / 3600.0)) / 1.852


[teleport] iter 2: dropped 48 points


...

,voyage_id,origin_call_id,dest_call_id,origin_port_id,dest_port_id,origin_port_name,dest_port_name,t_depart_origin,t_arrive_dest,total_waiting_sec,mmsi,n_points,max_dt_sec,max_dt_hr,gap_from,gap_to,large_time_gap_flag,total_loitering_sec,loitering_events_count
95,96,96,97,TWTPE,CNLYA,TWTPE,CNLYA,2025-11-09 15:30:14,2025-11-11 11:14:27,7327.0,477300400,1191,710.0,0.197222,2025-11-09 18:32:40,2025-11-09 18:44:30,False,102253.0,8
96,97,97,98,CNLYA,TWTPE,CNLYA,TWTPE,2025-11-12 09:10:21,2025-11-14 09:52:57,0.0,477300400,1296,7813.0,2.170278,2025-11-14 07:42:44,2025-11-14 09:52:57,False,90480.0,20
14,15,15,16,CNFOC,TWTPE,CNFOC,TWTPE,2025-02-06 14:32:07,2025-02-08 06:28:07,21599.0,477300400,1197,362.0,0.100556,2025-02-07 16:32:07,2025-02-07 16:38:09,False,67440.0,16
8,9,9,10,CNFOC,TWTPE,CNFOC,TWTPE,2025-01-16 07:08:09,2025-01-18 03:00:03,22919.0,477300400,639,25078.0,6.966111,2025-01-16 15:36:05,2025-01-16 22:34:03,True,49914.0,22
57,58,58,59,TWTPE,CNLYA,TWTPE,CNLYA,2025-07-05 06:28:34,2025-07-08 13:20:38,89756.0,477300400,2200,5634.0,1.565000,2025-07-07 03:26:41,2025-07-07 05:00:35,False,49444.0,22
89,90,90,91,TWTPE,CNNDE,TWTPE,CNNDE,2025-10-23 11:26:39,2025-10-24 12:18:12,0.0,477300400,611,1314.0,0.365000,2025-10-24 00:00:15,2025-10-24 00:22:09,False,46173.0,12
18,19,19,20,CNNDE,CNLYA,CNNDE,CNLYA,2025-03-06 13:00:16,2025-03-07 09:10:13,7801.0,477300400,599,721.0,0.200278,2025-03-07 07:16:13,2025-03-07 07:28:14,False,42837.0,12
17,18,18,19,TWTPE,CNNDE,TWTPE,CNNDE,2025-02-18 06:12:12,2025-02-19 12:40:39,9243.0,477300400,897,616.0,0.171111,2025-02-18 10:18:36,2025-02-18 10:28:52,False,32667.0,12
15,16,16,17,TWTPE,CNFOC,TWTPE,CNFOC,2025-02-12 17:30:13,2025-02-13 15:36:09,2032.0,477300400,652,275.0,0.076389,2025-02-12 17:44:10,2025-02-12 17:48:45,False,31796.0,5
19,20,20,21,CNLYA,TWTPE,CNLYA,TWTPE,2025-03-08 10:38:13,2025-03-09 08:32:13,0.0,477300400,656,360.0,0.100000,2025-03-08 18:12:16,2025-03-08 18:18:16,False,30120.0,4


edge-guard(前後3點) before: 455
edge-guard(前後3點) kept  : 294
edge-guard(前後3點) drop  : 161


,voyage_id,total_loitering_sec,loitering_events_count
0,1,1560.0,2
1,2,0.0,0
2,3,0.0,0
3,4,10320.0,7
4,5,0.0,0


,voyage_id,t_start,t_end,duration_sec,merged_from_event_ids,event_id,anomaly_type,n_points,r95_km
0,1,2025-01-01 12:17:19,2025-01-01 12:31:19,840.0,[2],1,loitering_merged,8,1.255442
1,1,2025-01-01 22:35:19,2025-01-01 22:47:19,720.0,[3],2,loitering_merged,7,1.167740
2,4,2025-01-07 16:15:42,2025-01-08 06:09:42,50040.0,"[7, 8, 9, 10, 11, 12, 13]",3,loitering_merged,341,0.101904
3,6,2025-01-12 08:58:00,2025-01-12 09:14:00,960.0,[15],4,loitering_merged,8,0.010450
4,7,2025-01-13 21:56:06,2025-01-13 22:36:06,2400.0,"[17, 18]",5,loitering_merged,11,2.092467
5,9,2025-01-16 23:46:09,2025-01-17 23:34:09,85680.0,"[20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 3...",6,loitering_merged,519,0.455727
6,9,2025-01-18 00:12:09,2025-01-18 02:12:09,7200.0,"[37, 38, 39, 40]",7,loitering_merged,60,5.208899
7,10,2025-01-21 23:32:07,2025-01-22 03:12:07,13200.0,"[43, 44, 45]",8,loitering_merged,108,9.452717
8,10,2025-01-22 04:58:07,2025-01-22 05:16:07,1080.0,[46],9,loitering_merged,10,0.863983
9,12,2025-01-25 10:26:04,2025-01-25 11:20:04,3240.0,[50],10,loitering_merged,27,2.461402


,voyage_id,total_loitering_sec,loitering_events_count
0,1,1560.0,2
1,2,0.0,0
2,3,0.0,0
3,4,50040.0,1
4,5,0.0,0
5,6,960.0,1
6,7,2400.0,1
7,8,0.0,0
8,9,92880.0,2
9,10,14280.0,2


### Visualization & Validation

In [2]:
import folium
from folium import LayerControl
from branca.element import Element
import pandas as pd
import numpy as np

def visualize_voyages_with_loitering_and_anchorage(
    df,
    voyages_df,
    loitering_events,
    stops_labeled,          # 需要 stop_type, t_start, t_end
    ports_df,
    voyage_ids=None,
    max_voyages=10,
    output_html="outputs/voyages_loitering_anchorage_map.html",
    sailing_color="blue",
    anch_color="orange",
    loiter_color="red",
    zoom_start=6,
):
    # -----------------------
    # 0) Normalize inputs
    # -----------------------
    d = df.copy()
    d["Timestamp"] = pd.to_datetime(d["Timestamp"])

    v_all = voyages_df.copy().dropna(subset=["t_depart_origin", "t_arrive_dest"])
    v_all["t_depart_origin"] = pd.to_datetime(v_all["t_depart_origin"])
    v_all["t_arrive_dest"]   = pd.to_datetime(v_all["t_arrive_dest"])
    v_all = v_all.sort_values("t_depart_origin")

    if voyage_ids is not None:
        v_sel = v_all[v_all["voyage_id"].isin(voyage_ids)].copy()
    else:
        v_sel = v_all.head(max_voyages).copy()

    if v_sel.empty:
        print("[WARN] 沒有符合條件的 voyages 可以畫。")
        return None

    # --- loitering_events: ensure schema ---
    le = loitering_events.copy() if isinstance(loitering_events, pd.DataFrame) else pd.DataFrame()
    if le.empty:
        le = pd.DataFrame(columns=["voyage_id","t_start","t_end"])
    for c in ["voyage_id","t_start","t_end"]:
        if c not in le.columns:
            le[c] = pd.Series(dtype="object")

    le["voyage_id"] = pd.to_numeric(le["voyage_id"], errors="coerce")
    le["t_start"] = pd.to_datetime(le["t_start"], errors="coerce")
    le["t_end"]   = pd.to_datetime(le["t_end"], errors="coerce")
    le = le.dropna(subset=["voyage_id","t_start","t_end"])
    le = le[le["t_end"] > le["t_start"]]

    # --- stops_labeled: anchored/anchorage-like by time intervals ---
    st = stops_labeled.copy() if isinstance(stops_labeled, pd.DataFrame) else pd.DataFrame()
    if st.empty:
        st = pd.DataFrame(columns=["stop_type","t_start","t_end"])
    for c in ["stop_type","t_start","t_end"]:
        if c not in st.columns:
            st[c] = pd.Series(dtype="object")

    st["t_start"] = pd.to_datetime(st["t_start"], errors="coerce")
    st["t_end"]   = pd.to_datetime(st["t_end"], errors="coerce")
    st["stop_type"] = st["stop_type"].astype(str).str.lower()
    st = st.dropna(subset=["t_start","t_end"])
    st = st[st["t_end"] > st["t_start"]]

    # 同時支援 anchorage_like / anchored_like / anchored
    st_anch = st[st["stop_type"].isin(["anchorage_like", "anchored_like", "anchored"])].copy()

    # --- ports_df -> map (optional markers) ---
    p = ports_df.copy()
    if "port_id" not in p.columns and "unlocode" in p.columns:
        p = p.rename(columns={"unlocode":"port_id"})
    if "port_name" not in p.columns:
        if "name" in p.columns:
            p["port_name"] = p["name"]
        else:
            p["port_name"] = p.get("port_id", "PORT")

    port_map = {}
    need_cols = ["port_id","lat","lon","port_name"]
    if all(c in p.columns for c in need_cols):
        pp = p[need_cols].dropna(subset=["lat","lon"]).copy()
        for _, r in pp.iterrows():
            port_map[str(r["port_id"])] = (float(r["lat"]), float(r["lon"]), str(r["port_name"]))

    # -----------------------
    # 1) Map center
    # -----------------------
    mask_time = np.zeros(len(d), dtype=bool)
    for _, v in v_sel.iterrows():
        mask_time |= ((d["Timestamp"] >= v["t_depart_origin"]) & (d["Timestamp"] <= v["t_arrive_dest"]))
    d_sel = d[mask_time]

    center_lat = float(d_sel["Lat"].median()) if not d_sel.empty else float(d["Lat"].median())
    center_lon = float(d_sel["Long"].median()) if not d_sel.empty else float(d["Long"].median())
    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start, tiles="OpenStreetMap")

    def _merge_intervals(intervals):
        if not intervals:
            return []
        intervals = sorted(intervals, key=lambda x: x[0])
        out = [intervals[0]]
        for s,e in intervals[1:]:
            ps,pe = out[-1]
            if s <= pe:  # overlap / touch
                out[-1] = (ps, max(pe, e))
            else:
                out.append((s,e))
        return out

    # -----------------------
    # 2) Draw each voyage
    # -----------------------
    for _, v in v_sel.iterrows():
        voy_id = int(v["voyage_id"])
        t_dep  = v["t_depart_origin"]
        t_arr  = v["t_arrive_dest"]

        op = str(v.get("origin_port_id", v.get("origin_port_name", "")))
        dp = str(v.get("dest_port_id",   v.get("dest_port_name", "")))
        layer_name = f"Voyage {voy_id}: {op} → {dp}".strip()

        fg = folium.FeatureGroup(name=layer_name)

        seg = d[(d["Timestamp"] >= t_dep) & (d["Timestamp"] <= t_arr)].copy()
        seg = seg.sort_values("Timestamp")

        # 去掉 NaN Lat/Long，避免 folium 斷線或報錯
        seg["Lat"] = pd.to_numeric(seg["Lat"], errors="coerce")
        seg["Long"] = pd.to_numeric(seg["Long"], errors="coerce")
        seg = seg.dropna(subset=["Lat","Long"])
        if len(seg) < 2:
            continue

        seg["is_loiter"] = False
        seg["is_anch"] = False

        # (A) loitering intervals by voyage_id
        ev = le[le["voyage_id"] == voy_id].copy()
        lo_intervals = [(r["t_start"], r["t_end"]) for _, r in ev.iterrows()]
        lo_intervals = _merge_intervals([x for x in lo_intervals if x[1] > x[0]])
        for a,b in lo_intervals:
            seg.loc[(seg["Timestamp"] >= a) & (seg["Timestamp"] <= b), "is_loiter"] = True

        # (B) anchorage_like intervals by time overlap (does not require dest proximity)
        stv = st_anch[(st_anch["t_start"] <= t_arr) & (st_anch["t_end"] >= t_dep)].copy()
        an_intervals = [(r["t_start"], r["t_end"]) for _, r in stv.iterrows()]
        an_intervals = _merge_intervals([x for x in an_intervals if x[1] > x[0]])
        for a,b in an_intervals:
            seg.loc[(seg["Timestamp"] >= a) & (seg["Timestamp"] <= b), "is_anch"] = True

        # state priority: loiter(2) > anch(1) > sailing(0)
        seg["state"] = 0
        seg.loc[seg["is_anch"], "state"] = 1
        seg.loc[seg["is_loiter"], "state"] = 2

        def add_polyline(segment, state):
            if len(segment) < 2:
                return
            if state == 2:
                color, weight, opacity = loiter_color, 6, 0.95
            elif state == 1:
                color, weight, opacity = anch_color, 5, 0.90
            else:
                color, weight, opacity = sailing_color, 3, 0.75

            folium.PolyLine(
                locations=segment,
                color=color,
                weight=weight,
                opacity=opacity
            ).add_to(fg)

        # ✅ 修正斷線：state 切換時，新段用 [prev_pt, pt] 補上跨段邊
        coords = seg[["Lat","Long","state"]].to_numpy()
        cur_state = int(coords[0][2])
        prev_pt = (float(coords[0][0]), float(coords[0][1]))
        cur_seg = [prev_pt]

        for lat, lon, stt in coords[1:]:
            stt = int(stt)
            pt = (float(lat), float(lon))

            if stt == cur_state:
                cur_seg.append(pt)
            else:
                add_polyline(cur_seg, cur_state)
                cur_state = stt
                cur_seg = [prev_pt, pt]   # ★關鍵：補跨段邊

            prev_pt = pt

        add_polyline(cur_seg, cur_state)

        # start/end markers
        lat0, lon0 = float(seg["Lat"].iloc[0]), float(seg["Long"].iloc[0])
        lat1, lon1 = float(seg["Lat"].iloc[-1]), float(seg["Long"].iloc[-1])
        folium.CircleMarker([lat0, lon0], radius=5, color="green", fill=True, fill_opacity=0.9,
                            tooltip=f"Start voyage {voy_id}").add_to(fg)
        folium.CircleMarker([lat1, lon1], radius=5, color="black", fill=True, fill_opacity=0.9,
                            tooltip=f"End voyage {voy_id}").add_to(fg)

        # optional port markers
        o_id = str(v.get("origin_port_id", ""))
        d_id = str(v.get("dest_port_id", ""))
        if o_id in port_map:
            plat, plon, pname = port_map[o_id]
            folium.Marker(
                [plat, plon],
                popup=f"Origin {o_id} ({pname})",
                icon=folium.Icon(color="green", icon="ship", prefix="fa")
            ).add_to(fg)
        if d_id in port_map:
            plat, plon, pname = port_map[d_id]
            folium.Marker(
                [plat, plon],
                popup=f"Dest {d_id} ({pname})",
                icon=folium.Icon(color="darkred", icon="anchor", prefix="fa")
            ).add_to(fg)

        fg.add_to(m)

    # -----------------------
    # 3) LayerControl + toggle buttons
    # -----------------------
    LayerControl(collapsed=False).add_to(m)

    toggle_all = Element("""
    <script>
    (function() {
      function setAllOverlays(checked){
        var inputs = document.querySelectorAll(
          '.leaflet-control-layers-overlays input[type="checkbox"]'
        );
        inputs.forEach(function(input){
          if (input.checked !== checked){
            input.click();
          }
        });
      }
      function injectButtons(){
        var lc = document.querySelector('.leaflet-control-layers');
        if(!lc) return;
        if (lc.querySelector('.folium-layer-toggle-all')) return;

        var form = lc.querySelector('form') || lc;
        var box = document.createElement('div');
        box.className = 'folium-layer-toggle-all';
        box.style.padding = '6px 8px';
        box.style.borderBottom = '1px solid #ddd';
        box.style.display = 'flex';
        box.style.gap = '6px';

        var btnAll = document.createElement('button');
        btnAll.type = 'button';
        btnAll.textContent = '全選';
        btnAll.style.cursor = 'pointer';

        var btnNone = document.createElement('button');
        btnNone.type = 'button';
        btnNone.textContent = '全不選';
        btnNone.style.cursor = 'pointer';

        btnAll.onclick = function(){ setAllOverlays(true); };
        btnNone.onclick = function(){ setAllOverlays(false); };

        box.appendChild(btnAll);
        box.appendChild(btnNone);
        form.insertBefore(box, form.firstChild);
      }
      window.addEventListener('load', injectButtons);
    })();
    </script>
    """)
    m.get_root().html.add_child(toggle_all)

    # save & open
    import os
    from pathlib import Path
    outp = Path(output_html)
    outp.parent.mkdir(parents=True, exist_ok=True)
    m.save(str(outp))
    print(f"已輸出地圖到 {outp.resolve()}")

    try:
        import webbrowser
        webbrowser.open(outp.resolve().as_uri())
    except Exception:
        pass

    return m


## 生成 乾淨 ETA 點、Ref Route 剔除點(loiter, port_call, anchorage_like, pr 10)

In [16]:
# ============================================================
# Cell 1 — Step 1~4: Build clean sailing points + filters
#   1) drop large_time_gap voyages
#   2) remove anchorage_like + loitering + port_call(core shrink) points
#   3) OD-level trim (drop top/bottom 10% by clean distance)
#   4) quality gates by clean_ratio / clean_duration_ratio / min_clean_points
# Outputs:
#   voyage_clean_summary
#   df_clean_points_all
#   voyages_ref_pool
#   df_clean_points_ref_pool
# ============================================================

import numpy as np
import pandas as pd
from math import radians, sin, cos, atan2, sqrt

# --------------------------
# Config (tune as needed)
# --------------------------
CFG = dict(
    # step1 hard drops
    require_no_large_time_gap=True,
    min_raw_points=30,

    # step2 cleaning intervals
    # port_call core is removed, but keep edges for "in/out port sailing"
    portcall_keep_edge_min=20,  # minutes kept on BOTH sides; core is removed if longer than 2*edge

    # step3 OD trim
    od_trim_frac=0.10,
    od_trim_min_n=10,           # only do trim if OD group has >= this many voyages

    # step4 quality gates (where "V3 only half" gets removed)
    min_clean_points=30,
    min_clean_ratio_dist=0.55,   # clean_km / raw_km
    min_clean_duration_ratio=0.55, # (raw_duration - removed_intervals_duration)/raw_duration

    # optional sanity
    min_clean_km=10.0,          # too short is not informative (optional)
)

# --------------------------
# helpers
# --------------------------
def _to_lon180(dlon):
    return ((dlon + 180.0) % 360.0) - 180.0

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlmb = radians(_to_lon180(lon2 - lon1))
    a = sin(dphi/2)**2 + cos(p1)*cos(p2)*sin(dlmb/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1-a))

def path_len_km(lat, lon):
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    if len(lat) < 2:
        return 0.0
    tot = 0.0
    for i in range(1, len(lat)):
        if np.isnan(lat[i-1]) or np.isnan(lon[i-1]) or np.isnan(lat[i]) or np.isnan(lon[i]):
            continue
        tot += haversine_km(lat[i-1], lon[i-1], lat[i], lon[i])
    return float(tot)

def merge_intervals(intervals):
    """intervals: list of (start_ts, end_ts), return merged non-overlap sorted"""
    if not intervals:
        return []
    intervals = sorted(intervals, key=lambda x: x[0])
    out = [[intervals[0][0], intervals[0][1]]]
    for s,e in intervals[1:]:
        if s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s,e])
    return [(a,b) for a,b in out]

def clip_intervals_to_window(intervals, t0, t1):
    """clip to [t0,t1] and drop invalid"""
    out = []
    for a,b in intervals:
        aa = max(a, t0)
        bb = min(b, t1)
        if bb > aa:
            out.append((aa,bb))
    return merge_intervals(out)

def sum_overlap_sec(intervals, t0, t1):
    """intervals already merged/clipped ok, sum duration within [t0,t1]"""
    tot = 0.0
    for a,b in clip_intervals_to_window(intervals, t0, t1):
        tot += (b-a).total_seconds()
    return float(tot)

def mask_out_intervals(ts: pd.Series, intervals):
    """return keep mask: True if timestamp not in any interval"""
    if ts.empty:
        return np.array([], dtype=bool)
    keep = np.ones(len(ts), dtype=bool)
    if not intervals:
        return keep
    t = ts.to_numpy()
    for a,b in intervals:
        keep &= ~((t >= np.datetime64(a)) & (t <= np.datetime64(b)))
    return keep

def compute_cumdist_and_p(seg: pd.DataFrame, lat_col="Lat", lon_col="Long"):
    """seg must be sorted by time and have Lat/Long numeric; returns seg with step_km/cum_km/p/total_clean_km"""
    out = seg.copy()
    lat = out[lat_col].to_numpy(dtype=float)
    lon = out[lon_col].to_numpy(dtype=float)
    step = np.zeros(len(out), dtype=float)
    for i in range(1, len(out)):
        if np.isnan(lat[i-1]) or np.isnan(lon[i-1]) or np.isnan(lat[i]) or np.isnan(lon[i]):
            step[i] = np.nan
        else:
            step[i] = haversine_km(lat[i-1], lon[i-1], lat[i], lon[i])
    step = np.nan_to_num(step, nan=0.0, posinf=0.0, neginf=0.0)
    cum = np.cumsum(step)
    total = float(cum[-1]) if len(cum) else 0.0
    p = (cum / total) if total > 0 else np.zeros(len(out), dtype=float)

    out["step_km_clean"] = step
    out["cum_km_clean"] = cum
    out["p_clean"] = p
    out["total_clean_km"] = total
    return out

# --------------------------
# 0) normalize inputs
# --------------------------
d = df.copy()
d["Timestamp"] = pd.to_datetime(d["Timestamp"], errors="coerce")
d["Lat"] = pd.to_numeric(d["Lat"], errors="coerce")
d["Long"] = pd.to_numeric(d["Long"], errors="coerce")
d = d.dropna(subset=["Timestamp"]).sort_values("Timestamp")

v0 = voyages_with_waiting.copy()
v0["t_depart_origin"] = pd.to_datetime(v0["t_depart_origin"], errors="coerce")
v0["t_arrive_dest"]   = pd.to_datetime(v0["t_arrive_dest"], errors="coerce")
v0["origin_port_id"]  = v0["origin_port_id"].astype(str)
v0["dest_port_id"]    = v0["dest_port_id"].astype(str)

# stops -> intervals
st = stops_labeled.copy() if isinstance(stops_labeled, pd.DataFrame) else pd.DataFrame()
if not st.empty:
    st["t_start"] = pd.to_datetime(st["t_start"], errors="coerce")
    st["t_end"]   = pd.to_datetime(st["t_end"], errors="coerce")
    st["stop_type"] = st["stop_type"].astype(str).str.lower()
    st = st.dropna(subset=["t_start","t_end"])
    st = st[st["t_end"] > st["t_start"]]

st_anch = st[st["stop_type"].isin(["anchorage_like","anchored_like","anchored"])].copy() if not st.empty else st
st_port = st[st["stop_type"].isin(["port_call"])].copy() if not st.empty else st

# loiter events pick best available variable
if "loitering_events_merged" in globals() and isinstance(loitering_events_merged, pd.DataFrame):
    le = loitering_events_merged.copy()
elif "loitering_events_kept_3pts" in globals() and isinstance(loitering_events_kept_3pts, pd.DataFrame):
    le = loitering_events_kept_3pts.copy()
elif "loitering_events" in globals() and isinstance(loitering_events, pd.DataFrame):
    le = loitering_events.copy()
else:
    le = pd.DataFrame(columns=["voyage_id","t_start","t_end"])

if not le.empty:
    for c in ["voyage_id","t_start","t_end"]:
        if c not in le.columns:
            le[c] = pd.Series(dtype="object")
    le["voyage_id"] = pd.to_numeric(le["voyage_id"], errors="coerce")
    le["t_start"] = pd.to_datetime(le["t_start"], errors="coerce")
    le["t_end"]   = pd.to_datetime(le["t_end"], errors="coerce")
    le = le.dropna(subset=["voyage_id","t_start","t_end"])
    le = le[le["t_end"] > le["t_start"]]
else:
    le = pd.DataFrame(columns=["voyage_id","t_start","t_end"])

# --------------------------
# Step 1~2 per-voyage: build clean points + metrics
# --------------------------
rows_summary = []
clean_points_list = []

keep_edge = pd.Timedelta(minutes=float(CFG["portcall_keep_edge_min"]))

for _, vr in v0.iterrows():
    vid = int(vr["voyage_id"])
    t0 = vr["t_depart_origin"]
    t1 = vr["t_arrive_dest"]
    if pd.isna(t0) or pd.isna(t1) or t1 <= t0:
        continue

    # (Step1) hard drop large gap
    if CFG["require_no_large_time_gap"] and bool(vr.get("large_time_gap_flag", False)):
        rows_summary.append({
            "voyage_id": vid,
            "origin_port_id": vr["origin_port_id"],
            "dest_port_id": vr["dest_port_id"],
            "drop_reason": "large_time_gap_flag",
        })
        continue

    seg_raw = d[(d["Timestamp"] >= t0) & (d["Timestamp"] <= t1)].copy()
    seg_raw = seg_raw.sort_values("Timestamp").dropna(subset=["Lat","Long"])
    raw_points = int(len(seg_raw))
    raw_duration_sec = float((t1 - t0).total_seconds())
    raw_dist_km = path_len_km(seg_raw["Lat"].to_numpy(), seg_raw["Long"].to_numpy()) if raw_points >= 2 else 0.0

    if raw_points < int(CFG["min_raw_points"]):
        rows_summary.append({
            "voyage_id": vid, "origin_port_id": vr["origin_port_id"], "dest_port_id": vr["dest_port_id"],
            "raw_points": raw_points, "raw_duration_sec": raw_duration_sec, "raw_dist_km": raw_dist_km,
            "drop_reason": "min_raw_points",
        })
        continue

    # build removal intervals within voyage window
    intervals = []

    # anchorage intervals (time overlap, no voyage_id needed)
    if not st_anch.empty:
        hit = st_anch[(st_anch["t_start"] <= t1) & (st_anch["t_end"] >= t0)]
        intervals += [(r["t_start"], r["t_end"]) for _, r in hit.iterrows()]

    # loitering intervals (prefer voyage_id match)
    if not le.empty:
        hit = le[le["voyage_id"] == vid]
        intervals += [(r["t_start"], r["t_end"]) for _, r in hit.iterrows()]

    # port_call core-shrink (keep edges)
    if not st_port.empty and CFG["portcall_keep_edge_min"] > 0:
        hit = st_port[(st_port["t_start"] <= t1) & (st_port["t_end"] >= t0)]
        for _, r in hit.iterrows():
            a = r["t_start"] + keep_edge
            b = r["t_end"] - keep_edge
            if pd.notna(a) and pd.notna(b) and b > a:
                intervals.append((a,b))

    intervals = clip_intervals_to_window(merge_intervals(intervals), t0, t1)
    removed_time_sec = sum_overlap_sec(intervals, t0, t1)
    clean_duration_ratio = (raw_duration_sec - removed_time_sec) / raw_duration_sec if raw_duration_sec > 0 else np.nan
    clean_duration_ratio = float(np.clip(clean_duration_ratio, 0.0, 1.0)) if np.isfinite(clean_duration_ratio) else np.nan

    # remove points inside intervals
    if seg_raw.empty:
        clean_points = 0
        clean_dist_km = 0.0
        clean_ratio_dist = np.nan
        seg_clean = seg_raw.copy()
    else:
        keep_mask = mask_out_intervals(seg_raw["Timestamp"], intervals)
        seg_clean = seg_raw.loc[keep_mask].copy()
        seg_clean = seg_clean.sort_values("Timestamp").dropna(subset=["Lat","Long"])
        clean_points = int(len(seg_clean))
        clean_dist_km = path_len_km(seg_clean["Lat"].to_numpy(), seg_clean["Long"].to_numpy()) if clean_points >= 2 else 0.0
        clean_ratio_dist = (clean_dist_km / raw_dist_km) if raw_dist_km > 0 else np.nan

    # compute p_clean + cumdist on seg_clean
    if clean_points >= 2:
        seg_clean2 = compute_cumdist_and_p(seg_clean, lat_col="Lat", lon_col="Long")
        seg_clean2["voyage_id"] = vid
        seg_clean2["origin_port_id"] = vr["origin_port_id"]
        seg_clean2["dest_port_id"] = vr["dest_port_id"]
        clean_points_list.append(seg_clean2)
        total_clean_km = float(seg_clean2["total_clean_km"].iloc[0])
    else:
        total_clean_km = float(clean_dist_km)

    rows_summary.append({
        "voyage_id": vid,
        "origin_port_id": vr["origin_port_id"],
        "dest_port_id": vr["dest_port_id"],
        "t_depart_origin": t0,
        "t_arrive_dest": t1,

        "raw_points": raw_points,
        "raw_duration_sec": raw_duration_sec,
        "raw_dist_km": float(raw_dist_km),

        "clean_points": clean_points,
        "clean_dist_km": float(clean_dist_km),
        "total_clean_km": float(total_clean_km),

        "clean_ratio_dist": float(clean_ratio_dist) if np.isfinite(clean_ratio_dist) else np.nan,
        "removed_time_sec": float(removed_time_sec),
        "clean_duration_ratio": clean_duration_ratio,

        "drop_reason": "",
    })

voyage_clean_summary = pd.DataFrame(rows_summary)

# attach OD label
if not voyage_clean_summary.empty:
    voyage_clean_summary["od"] = (
        voyage_clean_summary["origin_port_id"].astype(str) + "→" + voyage_clean_summary["dest_port_id"].astype(str)
    )

df_clean_points_all = pd.concat(clean_points_list, ignore_index=True) if clean_points_list else pd.DataFrame()

print("[Cell1] voyage_clean_summary rows:", len(voyage_clean_summary))
print("[Cell1] df_clean_points_all rows:", len(df_clean_points_all))

# --------------------------
# Step 3: OD-level trim (clean distance)
# --------------------------
voyage_clean_summary["od_trim_dropped"] = False

if not voyage_clean_summary.empty:
    for (o,dest), g in voyage_clean_summary.groupby(["origin_port_id","dest_port_id"], dropna=False):
        # only apply to rows that are not already dropped
        gg = g[g["drop_reason"].fillna("") == ""].copy()
        n = len(gg)
        if n < int(CFG["od_trim_min_n"]):
            continue
        k = max(1, int(np.round(n * float(CFG["od_trim_frac"]))))
        if 2*k >= n:
            continue

        gg = gg.sort_values("clean_dist_km")
        drop_ids = set(gg.head(k)["voyage_id"].tolist() + gg.tail(k)["voyage_id"].tolist())
        voyage_clean_summary.loc[voyage_clean_summary["voyage_id"].isin(drop_ids), "od_trim_dropped"] = True

# --------------------------
# Step 4: quality gates
# --------------------------
def _pass_quality(r):
    if str(r.get("drop_reason","")) != "":
        return False
    if bool(r.get("od_trim_dropped", False)):
        return False
    if int(r.get("clean_points", 0)) < int(CFG["min_clean_points"]):
        return False
    if float(r.get("clean_dist_km", 0.0)) < float(CFG["min_clean_km"]):
        return False
    crd = r.get("clean_ratio_dist", np.nan)
    crr = r.get("clean_duration_ratio", np.nan)
    if not np.isfinite(crd) or crd < float(CFG["min_clean_ratio_dist"]):
        return False
    if not np.isfinite(crr) or crr < float(CFG["min_clean_duration_ratio"]):
        return False
    return True

voyage_clean_summary["use_for_ref_train"] = voyage_clean_summary.apply(_pass_quality, axis=1)

# create final pools
voyages_ref_pool = v0.merge(
    voyage_clean_summary[["voyage_id","use_for_ref_train","od_trim_dropped","drop_reason",
                         "clean_points","clean_dist_km","clean_ratio_dist","clean_duration_ratio"]],
    on="voyage_id", how="left"
)
voyages_ref_pool = voyages_ref_pool[voyages_ref_pool["use_for_ref_train"] == True].copy()

if not df_clean_points_all.empty:
    df_clean_points_ref_pool = df_clean_points_all.merge(
        voyage_clean_summary[["voyage_id","use_for_ref_train"]],
        on="voyage_id", how="left"
    )
    df_clean_points_ref_pool = df_clean_points_ref_pool[df_clean_points_ref_pool["use_for_ref_train"] == True].copy()
    df_clean_points_ref_pool = df_clean_points_ref_pool.drop(columns=["use_for_ref_train"], errors="ignore")
else:
    df_clean_points_ref_pool = pd.DataFrame()

print("[Cell1] voyages_ref_pool:", len(voyages_ref_pool))
print("[Cell1] df_clean_points_ref_pool:", len(df_clean_points_ref_pool))

display(voyage_clean_summary.sort_values(["origin_port_id","dest_port_id","clean_dist_km"], ascending=[True,True,False]).head(10))


[Cell1] voyage_clean_summary rows: 99
[Cell1] df_clean_points_all rows: 50389
[Cell1] voyages_ref_pool: 69
[Cell1] df_clean_points_ref_pool: 43143


,voyage_id,origin_port_id,dest_port_id,t_depart_origin,t_arrive_dest,raw_points,raw_duration_sec,raw_dist_km,clean_points,clean_dist_km,total_clean_km,clean_ratio_dist,removed_time_sec,clean_duration_ratio,drop_reason,od,od_trim_dropped,use_for_ref_train
50,51,CNFOC,TWTPE,2025-06-07 06:12:25,2025-06-08 08:16:23,697.0,93838.0,272.632046,432.0,262.129506,262.129506,0.961477,36960.0,0.606130,,CNFOC→TWTPE,True,False
14,15,CNFOC,TWTPE,2025-02-06 14:32:07,2025-02-08 06:28:07,1197.0,143760.0,261.849234,438.0,252.017401,252.017401,0.962452,91320.0,0.364775,,CNFOC→TWTPE,True,False
98,99,CNFOC,TWTPE,2025-11-17 06:58:14,2025-11-18 02:00:43,477.0,68549.0,272.450689,356.0,251.361843,251.361843,0.922596,17760.0,0.740915,,CNFOC→TWTPE,True,False
88,89,CNFOC,TWTPE,2025-10-18 06:42:20,2025-10-19 04:58:40,585.0,80180.0,273.448330,327.0,247.326209,247.326209,0.904471,33960.0,0.576453,,CNFOC→TWTPE,False,True
41,42,CNFOC,TWTPE,2025-05-13 08:48:17,2025-05-13 22:58:15,193.0,50998.0,243.772567,188.0,243.749661,243.749661,0.999906,720.0,0.985882,,CNFOC→TWTPE,False,True
12,13,CNFOC,TWTPE,2025-01-26 07:02:07,2025-01-26 21:00:04,420.0,50277.0,242.489540,420.0,242.489540,242.489540,1.000000,0.0,1.000000,,CNFOC→TWTPE,False,True
60,61,CNFOC,TWTPE,2025-07-14 10:46:47,2025-07-14 23:20:35,290.0,45228.0,239.736934,280.0,239.494704,239.494704,0.998990,1560.0,0.965508,,CNFOC→TWTPE,False,True
86,87,CNFOC,TWTPE,2025-10-11 11:02:11,2025-10-12 02:06:13,423.0,54242.0,244.372372,343.0,239.237707,239.237707,0.978988,9960.0,0.816378,,CNFOC→TWTPE,False,True
76,77,CNFOC,TWTPE,2025-09-20 07:50:05,2025-09-20 20:18:13,315.0,44888.0,238.973197,315.0,238.973197,238.973197,1.000000,0.0,1.000000,,CNFOC→TWTPE,False,True
31,32,CNFOC,TWTPE,2025-04-11 08:06:09,2025-04-11 23:16:16,411.0,54607.0,242.067299,316.0,238.857941,238.857941,0.986742,12360.0,0.773655,,CNFOC→TWTPE,False,True


### 累積距離正規化

In [17]:
# ============================================================
# Cell 2 — Step 5: Build model ref routes per OD
#   Each voyage already has p_clean in df_clean_points_ref_pool
# Outputs:
#   ref_routes_df     (OD-level meta)
#   ref_polylines     dict[(origin,dest)] -> DataFrame with p/Lat/Long/coverage_n
# ============================================================

import numpy as np
import pandas as pd

REF_CFG = dict(
    n_grid=200,
    min_coverage_abs=2,
    min_coverage_frac=0.30,   # at least 30% voyages must cover that p-bin
    agg="median",             # "median" only for now
)

def _unwrap_lon_deg(lon_deg: np.ndarray) -> np.ndarray:
    """unwrap lon series to avoid 179/-179 median issues (simple unwrap in radians)."""
    lon = np.asarray(lon_deg, dtype=float)
    if len(lon) == 0:
        return lon
    rad = np.deg2rad(lon)
    rad_un = np.unwrap(rad)
    return np.rad2deg(rad_un)

def _interp_with_nan(p_src, x_src, p_grid):
    """np.interp but set outside range -> nan"""
    p_src = np.asarray(p_src, dtype=float)
    x_src = np.asarray(x_src, dtype=float)
    ok = np.isfinite(p_src) & np.isfinite(x_src)
    if ok.sum() < 2:
        return np.full_like(p_grid, np.nan, dtype=float)
    p = p_src[ok]
    x = x_src[ok]
    # ensure monotonic
    order = np.argsort(p)
    p = p[order]
    x = x[order]

    # drop duplicate p (keep last)
    _, idx = np.unique(p, return_index=True)
    p = p[idx]
    x = x[idx]
    if len(p) < 2:
        return np.full_like(p_grid, np.nan, dtype=float)

    xi = np.interp(p_grid, p, x)  # fills edges with endpoint -> we will nan-out
    xi[(p_grid < p.min()) | (p_grid > p.max())] = np.nan
    return xi

p_grid = np.linspace(0.0, 1.0, int(REF_CFG["n_grid"]))

ref_rows = []
ref_polylines = {}

if df_clean_points_ref_pool.empty:
    print("[Cell2] WARNING: df_clean_points_ref_pool is empty -> no ref can be built.")
    ref_routes_df = pd.DataFrame(columns=["origin_port_id","dest_port_id","n_voyages_used"])
else:
    # group points by OD
    for (o, d), g_od in df_clean_points_ref_pool.groupby(["origin_port_id","dest_port_id"]):
        vids = sorted(g_od["voyage_id"].dropna().unique().tolist())
        n_v = len(vids)
        if n_v == 0:
            continue

        lat_mat = []
        lon_mat = []

        for vid in vids:
            gv = g_od[g_od["voyage_id"] == vid].copy()
            gv = gv.sort_values("Timestamp")
            if len(gv) < 2:
                continue

            p = pd.to_numeric(gv["p_clean"], errors="coerce").to_numpy(dtype=float)
            lat = pd.to_numeric(gv["Lat"], errors="coerce").to_numpy(dtype=float)
            lon = pd.to_numeric(gv["Long"], errors="coerce").to_numpy(dtype=float)

            # unwrap lon per-voyage for stability
            lon_un = _unwrap_lon_deg(lon)

            lat_i = _interp_with_nan(p, lat, p_grid)
            lon_i = _interp_with_nan(p, lon_un, p_grid)

            lat_mat.append(lat_i)
            lon_mat.append(lon_i)

        if len(lat_mat) == 0:
            continue

        lat_mat = np.vstack(lat_mat)
        lon_mat = np.vstack(lon_mat)
        cov = np.sum(np.isfinite(lat_mat) & np.isfinite(lon_mat), axis=0)

        cov_th = max(int(REF_CFG["min_coverage_abs"]), int(np.ceil(REF_CFG["min_coverage_frac"] * lat_mat.shape[0])))

        if REF_CFG["agg"] == "median":
            lat_ref = np.nanmedian(lat_mat, axis=0)
            lon_ref = np.nanmedian(lon_mat, axis=0)
        else:
            lat_ref = np.nanmedian(lat_mat, axis=0)
            lon_ref = np.nanmedian(lon_mat, axis=0)

        # apply coverage gating
        lat_ref[cov < cov_th] = np.nan
        lon_ref[cov < cov_th] = np.nan

        ref_df = pd.DataFrame({
            "p": p_grid,
            "Lat": lat_ref,
            "Long": lon_ref,
            "coverage_n": cov,
        }).dropna(subset=["Lat","Long"])

        if len(ref_df) < 2:
            # too sparse ref; still record meta
            ref_rows.append({
                "origin_port_id": str(o),
                "dest_port_id": str(d),
                "n_voyages_used": int(lat_mat.shape[0]),
                "coverage_threshold": int(cov_th),
                "coverage_min": int(np.min(cov)) if len(cov) else 0,
                "coverage_mean": float(np.mean(cov)) if len(cov) else 0.0,
                "ref_select_reason": "model_ref_failed_sparse",
            })
            continue

        ref_polylines[(str(o), str(d))] = ref_df

        ref_rows.append({
            "origin_port_id": str(o),
            "dest_port_id": str(d),
            "n_voyages_used": int(lat_mat.shape[0]),
            "coverage_threshold": int(cov_th),
            "coverage_min": int(np.min(cov)) if len(cov) else 0,
            "coverage_mean": float(np.mean(cov)) if len(cov) else 0.0,
            "ref_select_reason": "model_ref_aggregate_clean_normdist",
        })

    ref_routes_df = pd.DataFrame(ref_rows).sort_values(["origin_port_id","dest_port_id"]).reset_index(drop=True)

print("[Cell2] Built ref ODs:", len(ref_routes_df))
print("[Cell2] ref_polylines keys:", len(ref_polylines))
display(ref_routes_df.head(10))


[Cell2] Built ref ODs: 18
[Cell2] ref_polylines keys: 7


,origin_port_id,dest_port_id,n_voyages_used,coverage_threshold,coverage_min,coverage_mean,ref_select_reason
0,CNFOC,TWTPE,20,6,20,20.0,model_ref_aggregate_clean_normdist
1,CNLYA,TWTPE,7,3,7,7.0,model_ref_aggregate_clean_normdist
2,CNNDE,TWKEL,1,2,1,1.0,model_ref_failed_sparse
3,JPMUR,TWTPE,1,2,1,1.0,model_ref_failed_sparse
4,JPNGO,TWTPE,3,2,3,3.0,model_ref_aggregate_clean_normdist
5,JPWAK,TWTPE,1,2,1,1.0,model_ref_failed_sparse
6,JPYKK,JPNGO,1,2,1,1.0,model_ref_failed_sparse
7,KRKAN,TWTPE,1,2,1,1.0,model_ref_failed_sparse
8,KRKPO,TWTPE,1,2,1,1.0,model_ref_failed_sparse
9,TWKEL,CNFOC,1,2,1,1.0,model_ref_failed_sparse


### 產生ETA 訓練資料

In [18]:
# ============================================================
# Cell 3 — Step 6: Build clean ETA training points
# Outputs:
#   df_eta_clean_points
# Notes:
#   - uses df_clean_points_ref_pool (already removed anch/loiter/portcall-core)
#   - computes eta_sec to arrival, res_clean_km, etc.
# ============================================================

import numpy as np
import pandas as pd

if df_clean_points_ref_pool.empty or voyages_ref_pool.empty:
    print("[Cell3] WARNING: empty ref_pool -> no ETA points produced.")
    df_eta_clean_points = pd.DataFrame()
else:
    # join arrival time / OD meta to points
    meta = voyages_ref_pool[[
        "voyage_id", "origin_port_id", "dest_port_id",
        "t_depart_origin", "t_arrive_dest",
        "origin_port_name", "dest_port_name",
        "large_time_gap_flag"
    ]].copy()

    meta["t_depart_origin"] = pd.to_datetime(meta["t_depart_origin"], errors="coerce")
    meta["t_arrive_dest"]   = pd.to_datetime(meta["t_arrive_dest"], errors="coerce")

    pts = df_clean_points_ref_pool.copy()
    pts["Timestamp"] = pd.to_datetime(pts["Timestamp"], errors="coerce")
    pts = pts.dropna(subset=["Timestamp"])

    df_eta_clean_points = pts.merge(meta, on=["voyage_id","origin_port_id","dest_port_id"], how="left")

    # compute ETA label
    df_eta_clean_points["eta_sec"] = (df_eta_clean_points["t_arrive_dest"] - df_eta_clean_points["Timestamp"]).dt.total_seconds()
    df_eta_clean_points = df_eta_clean_points[df_eta_clean_points["eta_sec"].notna()].copy()
    df_eta_clean_points = df_eta_clean_points[df_eta_clean_points["eta_sec"] >= 0].copy()

    # geometry progress features (based on clean distance normalization)
    df_eta_clean_points["p_clean"] = pd.to_numeric(df_eta_clean_points["p_clean"], errors="coerce")
    df_eta_clean_points["total_clean_km"] = pd.to_numeric(df_eta_clean_points["total_clean_km"], errors="coerce")
    df_eta_clean_points["cum_km_clean"] = pd.to_numeric(df_eta_clean_points["cum_km_clean"], errors="coerce")

    df_eta_clean_points["res_clean_km"] = (1.0 - df_eta_clean_points["p_clean"]) * df_eta_clean_points["total_clean_km"]
    df_eta_clean_points["eta_hr"] = df_eta_clean_points["eta_sec"] / 3600.0

    # some light cleanup
    keep_cols = [
        "voyage_id","origin_port_id","dest_port_id","origin_port_name","dest_port_name",
        "Timestamp","Lat","Long","Sog",
        "p_clean","cum_km_clean","total_clean_km","res_clean_km",
        "eta_sec","eta_hr",
        "t_depart_origin","t_arrive_dest",
    ]
    keep_cols = [c for c in keep_cols if c in df_eta_clean_points.columns]
    df_eta_clean_points = df_eta_clean_points[keep_cols].sort_values(["origin_port_id","dest_port_id","voyage_id","Timestamp"]).reset_index(drop=True)

    print("[Cell3] df_eta_clean_points rows:", len(df_eta_clean_points))
    display(df_eta_clean_points.head(10))


[Cell3] df_eta_clean_points rows: 43143


,voyage_id,origin_port_id,dest_port_id,origin_port_name,dest_port_name,Timestamp,Lat,Long,Sog,p_clean,cum_km_clean,total_clean_km,res_clean_km,eta_sec,eta_hr,t_depart_origin,t_arrive_dest
0,1,CNFOC,TWTPE,CNFOC,TWTPE,2025-01-01 09:57:19,26.008713,119.480165,0.9,0.000000,0.000000,237.855061,237.855061,49080.0,13.633333,2025-01-01 09:57:19,2025-01-01 23:35:19
1,1,CNFOC,TWTPE,CNFOC,TWTPE,2025-01-01 09:59:28,26.008182,119.481100,2.9,0.000465,0.110570,237.855061,237.744491,48951.0,13.597500,2025-01-01 09:57:19,2025-01-01 23:35:19
2,1,CNFOC,TWTPE,CNFOC,TWTPE,2025-01-01 10:01:18,26.006502,119.480722,2.2,0.001266,0.301166,237.855061,237.553896,48841.0,13.566944,2025-01-01 09:57:19,2025-01-01 23:35:19
3,1,CNFOC,TWTPE,CNFOC,TWTPE,2025-01-01 10:03:19,26.006202,119.480397,0.0,0.001462,0.347724,237.855061,237.507337,48720.0,13.533333,2025-01-01 09:57:19,2025-01-01 23:35:19
4,1,CNFOC,TWTPE,CNFOC,TWTPE,2025-01-01 10:05:19,26.006572,119.480730,1.5,0.001684,0.400662,237.855061,237.454400,48600.0,13.500000,2025-01-01 09:57:19,2025-01-01 23:35:19
5,1,CNFOC,TWTPE,CNFOC,TWTPE,2025-01-01 10:07:20,26.007688,119.481335,2.8,0.002265,0.538767,237.855061,237.316294,48479.0,13.466389,2025-01-01 09:57:19,2025-01-01 23:35:19
6,1,CNFOC,TWTPE,CNFOC,TWTPE,2025-01-01 10:09:19,26.009403,119.482138,3.7,0.003135,0.745676,237.855061,237.109385,48360.0,13.433333,2025-01-01 09:57:19,2025-01-01 23:35:19
7,1,CNFOC,TWTPE,CNFOC,TWTPE,2025-01-01 10:11:18,26.011633,119.482845,4.5,0.004219,1.003501,237.855061,236.851560,48241.0,13.400278,2025-01-01 09:57:19,2025-01-01 23:35:19
8,1,CNFOC,TWTPE,CNFOC,TWTPE,2025-01-01 10:13:18,26.013963,119.484303,5.1,0.005469,1.300760,237.855061,236.554302,48121.0,13.366944,2025-01-01 09:57:19,2025-01-01 23:35:19
9,1,CNFOC,TWTPE,CNFOC,TWTPE,2025-01-01 10:15:18,26.016010,119.486787,5.4,0.006884,1.637470,237.855061,236.217591,48001.0,13.333611,2025-01-01 09:57:19,2025-01-01 23:35:19


### Folium

In [19]:
import folium
from folium import LayerControl
from branca.element import Element
import pandas as pd
import numpy as np

def visualize_model_ref_routes_by_od(
    ref_routes_df: pd.DataFrame,
    ref_polylines: dict,
    ports_df: pd.DataFrame | None = None,
    od_list=None,                 # list of (origin_port_id, dest_port_id) to plot
    max_od: int = 30,
    output_html="outputs/model_ref_routes_map.html",
    zoom_start=6,
    add_port_markers=True,
    line_weight: int = 5,
    line_opacity: float = 0.9,
    show_coverage: bool = True,   # add small coverage tooltip on mid-point
):
    """
    Plot computed model ref routes on Folium.

    Parameters
    ----------
    ref_routes_df : DataFrame
        Must include at least: origin_port_id, dest_port_id, n_voyages_used (optional), ref_select_reason (optional).
    ref_polylines : dict
        {(origin_port_id, dest_port_id): DataFrame with columns ['Lat','Long'] and optional ['p','coverage_n'] }
    ports_df : DataFrame | None
        Optional ports table with columns like (port_id/unlocode, lat, lon, port_name/name).
    """

    # -------- normalize ports map (optional) --------
    port_map = {}
    if ports_df is not None and isinstance(ports_df, pd.DataFrame) and not ports_df.empty:
        p = ports_df.copy()
        if "port_id" not in p.columns and "unlocode" in p.columns:
            p = p.rename(columns={"unlocode": "port_id"})
        if "port_name" not in p.columns:
            if "name" in p.columns:
                p["port_name"] = p["name"]
            else:
                p["port_name"] = p.get("port_id", "PORT")
        need = ["port_id", "lat", "lon", "port_name"]
        if all(c in p.columns for c in need):
            pp = p[need].dropna(subset=["lat", "lon"]).copy()
            for _, r in pp.iterrows():
                port_map[str(r["port_id"])] = (float(r["lat"]), float(r["lon"]), str(r["port_name"]))

    # -------- normalize ref_routes_df --------
    rr = ref_routes_df.copy()
    if rr.empty:
        print("[WARN] ref_routes_df is empty.")
        return None

    rr["origin_port_id"] = rr["origin_port_id"].astype(str)
    rr["dest_port_id"]   = rr["dest_port_id"].astype(str)

    if od_list is not None:
        od_set = {(str(a), str(b)) for a, b in od_list}
        rr = rr[rr.apply(lambda x: (x["origin_port_id"], x["dest_port_id"]) in od_set, axis=1)].copy()
    else:
        rr = rr.head(max_od).copy()

    if rr.empty:
        print("[WARN] No ODs selected.")
        return None

    # -------- find map center from first valid polyline --------
    first_key = None
    for _, r in rr.iterrows():
        k = (str(r["origin_port_id"]), str(r["dest_port_id"]))
        if k in ref_polylines and isinstance(ref_polylines[k], pd.DataFrame) and len(ref_polylines[k]) >= 2:
            first_key = k
            break
    if first_key is None:
        print("[WARN] ref_polylines contains no drawable polyline for selected ODs.")
        return None

    seg0 = ref_polylines[first_key].copy()
    seg0["Lat"]  = pd.to_numeric(seg0["Lat"], errors="coerce")
    seg0["Long"] = pd.to_numeric(seg0["Long"], errors="coerce")
    seg0 = seg0.dropna(subset=["Lat", "Long"])
    if seg0.empty:
        print("[WARN] First polyline is empty after cleaning.")
        return None

    center_lat = float(seg0["Lat"].median())
    center_lon = float(seg0["Long"].median())
    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start, tiles="OpenStreetMap")

    # -------- draw each OD model ref route --------
    for _, r in rr.iterrows():
        o = str(r["origin_port_id"]); d = str(r["dest_port_id"])
        key = (o, d)
        if key not in ref_polylines:
            continue

        seg = ref_polylines[key].copy()
        if seg is None or seg.empty or len(seg) < 2:
            continue

        seg["Lat"]  = pd.to_numeric(seg["Lat"], errors="coerce")
        seg["Long"] = pd.to_numeric(seg["Long"], errors="coerce")
        seg = seg.dropna(subset=["Lat", "Long"])
        if len(seg) < 2:
            continue

        n_used = r.get("n_voyages_used", None)
        reason = str(r.get("ref_select_reason", "")).strip()

        layer_parts = [f"MODEL REF {o} → {d}"]
        if pd.notna(n_used):
            layer_parts.append(f"n={int(n_used)}")
        if reason:
            layer_parts.append(reason)
        layer_name = " | ".join(layer_parts)

        fg = folium.FeatureGroup(name=layer_name)

        coords = list(zip(seg["Lat"].astype(float), seg["Long"].astype(float)))
        folium.PolyLine(coords, weight=line_weight, opacity=line_opacity).add_to(fg)

        # start/end markers
        lat0, lon0 = coords[0]
        lat1, lon1 = coords[-1]
        folium.CircleMarker([lat0, lon0], radius=5, color="green", fill=True, fill_opacity=0.9,
                            tooltip=f"REF start {o}").add_to(fg)
        folium.CircleMarker([lat1, lon1], radius=5, color="black", fill=True, fill_opacity=0.9,
                            tooltip=f"REF end {d}").add_to(fg)

        # coverage tooltip (mid point)
        if show_coverage and "coverage_n" in seg.columns:
            mid_i = len(seg) // 2
            cov = seg["coverage_n"].iloc[mid_i]
            p_mid = seg["p"].iloc[mid_i] if "p" in seg.columns else None
            msg = f"coverage_n={int(cov)}"
            if p_mid is not None and pd.notna(p_mid):
                msg += f", p~{float(p_mid):.2f}"
            folium.CircleMarker([coords[mid_i][0], coords[mid_i][1]],
                                radius=4, color="purple", fill=True, fill_opacity=0.7,
                                tooltip=msg).add_to(fg)

        # optional port markers
        if add_port_markers and port_map:
            if o in port_map:
                plat, plon, pname = port_map[o]
                folium.Marker([plat, plon], popup=f"Origin {o} ({pname})",
                              icon=folium.Icon(color="green", icon="ship", prefix="fa")).add_to(fg)
            if d in port_map:
                plat, plon, pname = port_map[d]
                folium.Marker([plat, plon], popup=f"Dest {d} ({pname})",
                              icon=folium.Icon(color="darkred", icon="anchor", prefix="fa")).add_to(fg)

        fg.add_to(m)

    # -------- LayerControl + toggle buttons --------
    LayerControl(collapsed=False).add_to(m)

    toggle_all = Element("""
    <script>
    (function() {
      function setAllOverlays(checked){
        var inputs = document.querySelectorAll(
          '.leaflet-control-layers-overlays input[type="checkbox"]'
        );
        inputs.forEach(function(input){
          if (input.checked !== checked){
            input.click();
          }
        });
      }
      function injectButtons(){
        var lc = document.querySelector('.leaflet-control-layers');
        if(!lc) return;
        if (lc.querySelector('.folium-layer-toggle-all')) return;

        var form = lc.querySelector('form') || lc;
        var box = document.createElement('div');
        box.className = 'folium-layer-toggle-all';
        box.style.padding = '6px 8px';
        box.style.borderBottom = '1px solid #ddd';
        box.style.display = 'flex';
        box.style.gap = '6px';

        function mkBtn(txt){
          var b = document.createElement('button');
          b.type = 'button';
          b.textContent = txt;
          b.style.cursor = 'pointer';
          return b;
        }

        var btnAll = mkBtn('全選');
        var btnNone = mkBtn('全不選');
        btnAll.onclick = function(){ setAllOverlays(true); };
        btnNone.onclick = function(){ setAllOverlays(false); };

        box.appendChild(btnAll);
        box.appendChild(btnNone);
        form.insertBefore(box, form.firstChild);
      }
      window.addEventListener('load', injectButtons);
    })();
    </script>
    """)
    m.get_root().html.add_child(toggle_all)

    # save & open
    from pathlib import Path
    outp = Path(output_html)
    outp.parent.mkdir(parents=True, exist_ok=True)
    m.save(str(outp))
    print(f"已輸出地圖到 {outp.resolve()}")

    try:
        import webbrowser
        webbrowser.open(outp.resolve().as_uri())
    except Exception:
        pass

    return m


In [20]:
m = visualize_model_ref_routes_by_od(
    ref_routes_df=ref_routes_df,
    ref_polylines=ref_polylines,
    ports_df=ports_df,
    max_od=30,
    output_html="outputs/model_ref_routes_map.html",
    zoom_start=6,
)
m


已輸出地圖到 C:\Users\slab\Desktop\Slab Project\Stage2 ETA\outputs\model_ref_routes_map.html


In [21]:
print("df_clean_points_ref_pool cols:", df_clean_points_ref_pool.columns.tolist())
display(df_clean_points_ref_pool.head(3))
print("n_rows:", len(df_clean_points_ref_pool),
      "n_OD:", df_clean_points_ref_pool.groupby(["origin_port_id","dest_port_id"]).ngroups)

print("ports_df cols:", ports_df.columns.tolist())
display(ports_df.head(3))


df_clean_points_ref_pool cols: ['orig_index', 'orig_index_old', 'orig_index_old', 'orig_index_old', 'orig_index_old', 'orig_index_old', 'orig_index_old', 'orig_index_old', 'Pky', 'IMO', 'MMSI', 'CallSign', 'FacNumber', 'ShipName_CH', 'ShipName_ENG', 'VesselLine', 'VesselType', 'VesselSeries', 'Engine_Revolutions', 'GroundSpeed', 'WaterSpeed', 'Draft', 'Destination', 'DestinationSTD', 'DestinationCode', 'ETA', 'ETA_UTC', 'Nav_Status', 'Lat', 'Long', 'Heading', 'foucs', 'DataSource', 'PzOrder', 'Course', 'Rot', 'Sog', 'DataSourceLastTime', 'Timestamp', 'VesselDigitalType', 'Enable', 'PSA', 'PSB', 'PSC', 'IsDel', 'CreateTime', 'LastUpdateUserID', 'LastUpdateTime', 'LastUpdatePage', 'B_Rudder', 'B_Depth', 'B_WindSpeed', 'B_WindAngle', 'B_RPM', 'Long_360', 'step_km_clean', 'cum_km_clean', 'p_clean', 'total_clean_km', 'voyage_id', 'origin_port_id', 'dest_port_id']


,orig_index,orig_index_old,orig_index_old,orig_index_old,orig_index_old,orig_index_old,orig_index_old,orig_index_old,Pky,IMO,...,B_WindAngle,B_RPM,Long_360,step_km_clean,cum_km_clean,p_clean,total_clean_km,voyage_id,origin_port_id,dest_port_id
0,296,296,296,296,296,296,296,296,261088,9252058,...,NaN,NaN,119.480165,0.000000,0.000000,0.000000,237.855061,1,CNFOC,TWTPE
1,297,297,297,297,297,297,297,297,261089,9252058,...,NaN,NaN,119.481100,0.110570,0.110570,0.000465,237.855061,1,CNFOC,TWTPE
2,298,298,298,298,298,298,298,298,261090,9252058,...,NaN,NaN,119.480722,0.190595,0.301166,0.001266,237.855061,1,CNFOC,TWTPE


n_rows: 43143 n_OD: 18
ports_df cols: ['name', 'port_id', 'port_type', 'lat', 'lon', 'port_name']


,name,port_id,port_type,lat,lon,port_name
0,DUBAI,AEDXB,Port,25.27754,55.29378,AEDXB
1,JUMEIRAH,AEJYH,Port,25.21042,55.24256,AEJYH
2,FUJAIRAH,AEFJR,Port,25.16122,56.36583,AEFJR


### KNN 試作 ref route 

In [22]:
# ============================================================
# Cell 2 — Step 5 (A方法): Build model ref by KNN corridor + shortest path
# Inputs:
#   df_clean_points_ref_pool  (points after step1-4 filtering, clean sailing points)
#   ports_df
# Outputs:
#   ref_routes_df    (OD-level meta)
#   ref_polylines    dict[(origin,dest)] -> DataFrame(seq, Lat, Long, support, kind)
# ============================================================

import numpy as np
import pandas as pd
import heapq
from math import radians, sin, cos, atan2, sqrt

# --------------------
# Config
# --------------------
REF_A_CFG = dict(
    # node construction (grid clustering in km)
    grid_km=1.0,             # 1km grid => nodes not too dense
    min_points_per_node=1,   # keep even sparse cells (we will filter by graph/connectivity later)

    # graph edges
    knn_k=8,                 # KNN edges per node (bidirectional)
    port_connect_k=6,        # connect port node to k nearest nodes
    alpha=0.7,               # density weight exponent for transition edges
    knn_penalty=4.0,         # KNN edge cost multiplier (bigger => less likely)
    port_penalty=2.0,        # port-connection edge cost multiplier

    # OD eligibility
    min_voyages_od=2,        # < 2 voyages -> too risky; set 1 if you really want all ODs
    min_nodes_od=30,         # too few nodes => ref unstable
    max_nodes_od=8000,       # safety guard (shouldn't hit with your data)

    # output
    simplify_drop_consecutive_dups=True,
)

# --------------------
# Utilities
# --------------------
def _to_lon180(lon):
    return ((lon + 180.0) % 360.0) - 180.0

def _to_lon360(lon):
    return lon % 360.0

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlmb = radians(_to_lon180(lon2 - lon1))
    a = sin(dphi/2)**2 + cos(p1)*cos(p2)*sin(dlmb/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1-a))

def _equirect_xy_km(lat, lon360, lat0):
    # x = lon*cos(lat0), y = lat
    x = lon360 * cos(radians(lat0)) * 111.320
    y = lat * 110.574
    return x, y

def _safe_ports_map(ports_df: pd.DataFrame):
    p = ports_df.copy()
    if "port_id" not in p.columns and "unlocode" in p.columns:
        p = p.rename(columns={"unlocode": "port_id"})
    # your ports_df has: port_id, lat, lon
    need = ["port_id", "lat", "lon"]
    if not all(c in p.columns for c in need):
        raise KeyError(f"ports_df missing columns, need {need}, got {p.columns.tolist()}")
    p = p.dropna(subset=["port_id","lat","lon"]).copy()
    p["port_id"] = p["port_id"].astype(str)
    p["lat"] = pd.to_numeric(p["lat"], errors="coerce")
    p["lon"] = pd.to_numeric(p["lon"], errors="coerce")
    p = p.dropna(subset=["lat","lon"])
    p["lon360"] = p["lon"].apply(_to_lon360)
    return {r["port_id"]:(float(r["lat"]), float(r["lon360"])) for _, r in p.iterrows()}

def _dijkstra(adj, src, dst):
    # adj: dict[u] -> list[(v, cost, meta)]
    INF = 1e30
    dist = {src: 0.0}
    prev = {}
    prev_edge = {}
    pq = [(0.0, src)]
    seen = set()
    while pq:
        du, u = heapq.heappop(pq)
        if u in seen:
            continue
        seen.add(u)
        if u == dst:
            break
        for v, w, meta in adj.get(u, []):
            nd = du + w
            if nd < dist.get(v, INF):
                dist[v] = nd
                prev[v] = u
                prev_edge[v] = meta
                heapq.heappush(pq, (nd, v))

    if dst not in dist:
        return None, None, None

    # reconstruct
    path = [dst]
    edges = []
    cur = dst
    while cur != src:
        pu = prev[cur]
        edges.append(prev_edge[cur])
        path.append(pu)
        cur = pu
    path = path[::-1]
    edges = edges[::-1]
    return path, edges, dist[dst]

def _try_ckdtree_knn(X, k):
    # returns idx matrix: (n, k) nearest indices (excluding self if possible)
    try:
        from scipy.spatial import cKDTree
        tree = cKDTree(X)
        kk = min(k+1, len(X))
        d, idx = tree.query(X, k=kk)
        # drop self (first one)
        if kk >= 2:
            idx = idx[:, 1:]
        else:
            idx = idx[:, :0]
        return idx
    except Exception:
        # brute-force fallback (OK if n is small)
        n = len(X)
        idx_out = []
        for i in range(n):
            dx = X - X[i]
            dd = np.sum(dx*dx, axis=1)
            order = np.argsort(dd)
            order = order[order != i][:k]
            idx_out.append(order)
        return np.vstack(idx_out) if idx_out else np.zeros((0,0), dtype=int)

# --------------------
# Core per-OD builder
# --------------------
def build_model_ref_for_one_od(g_od: pd.DataFrame, o: str, d: str, ports_map: dict, cfg: dict):
    # drop duplicated col names (you have many 'orig_index_old')
    g_od = g_od.loc[:, ~g_od.columns.duplicated()].copy()

    vids = sorted(g_od["voyage_id"].dropna().unique().tolist())
    if len(vids) < cfg["min_voyages_od"]:
        return None, dict(reason=f"skip: n_voyages<{cfg['min_voyages_od']}", n_voyages=len(vids))

    # choose lon for computation
    if "Long_360" in g_od.columns:
        lon360 = pd.to_numeric(g_od["Long_360"], errors="coerce")
    else:
        lon360 = pd.to_numeric(g_od["Long"], errors="coerce").apply(_to_lon360)

    lat = pd.to_numeric(g_od["Lat"], errors="coerce")
    t = pd.to_datetime(g_od["Timestamp"], errors="coerce")

    g_od = g_od.assign(_lat=lat, _lon360=lon360, _t=t).dropna(subset=["_lat","_lon360","_t","voyage_id"]).copy()
    if len(g_od) < 10:
        return None, dict(reason="skip: too few points", n_points=len(g_od), n_voyages=len(vids))

    lat0 = float(np.nanmedian(g_od["_lat"].to_numpy()))
    x, y = _equirect_xy_km(g_od["_lat"].to_numpy(), g_od["_lon360"].to_numpy(), lat0)

    # grid clustering
    grid = float(cfg["grid_km"])
    gx = np.floor(x / grid).astype(int)
    gy = np.floor(y / grid).astype(int)
    g_od["_cell"] = list(zip(gx, gy))

    # build nodes: median lat/lon per cell
    node_groups = g_od.groupby("_cell", sort=False)
    if node_groups.ngroups < cfg["min_nodes_od"]:
        return None, dict(reason=f"skip: n_nodes<{cfg['min_nodes_od']}", n_nodes=node_groups.ngroups, n_voyages=len(vids))

    if node_groups.ngroups > cfg["max_nodes_od"]:
        return None, dict(reason=f"skip: n_nodes>{cfg['max_nodes_od']}", n_nodes=node_groups.ngroups, n_voyages=len(vids))

    node_df = node_groups.agg(
        Lat=("_lat", "median"),
        Lon360=("_lon360", "median"),
        n_points=("_lat", "size"),
    ).reset_index(drop=False).rename(columns={"_cell":"cell"})
    node_df["node_id"] = np.arange(len(node_df), dtype=int)

    cell_to_node = {c:int(nid) for c, nid in zip(node_df["cell"], node_df["node_id"])}
    g_od["node_id"] = g_od["_cell"].map(cell_to_node).astype(int)

    # transition edge counts from voyage time ordering
    trans_cnt = {}
    for vid, gv in g_od.sort_values("_t").groupby("voyage_id"):
        seq = gv["node_id"].to_numpy(dtype=int)
        if cfg["simplify_drop_consecutive_dups"]:
            # compress consecutive duplicates
            keep = [seq[0]]
            for u in seq[1:]:
                if u != keep[-1]:
                    keep.append(u)
            seq = np.array(keep, dtype=int)
        if len(seq) < 2:
            continue
        for u, v in zip(seq[:-1], seq[1:]):
            if u == v:
                continue
            trans_cnt[(u, v)] = trans_cnt.get((u, v), 0) + 1

    # build adjacency
    adj = {int(u): [] for u in node_df["node_id"].tolist()}

    def add_edge(u, v, dist_km, kind, support):
        # cost: distance * (1/support^alpha) for transition; for knn/port use penalty
        if kind == "trans":
            a = float(cfg["alpha"])
            w = float(dist_km) / (float(support) ** a)
        elif kind == "knn":
            w = float(dist_km) * float(cfg["knn_penalty"])
        elif kind == "port":
            w = float(dist_km) * float(cfg["port_penalty"])
        else:
            w = float(dist_km)
        adj[u].append((v, w, dict(kind=kind, dist_km=float(dist_km), support=int(support))))

    # add transition edges
    lat_arr = node_df["Lat"].to_numpy(dtype=float)
    lon_arr = node_df["Lon360"].to_numpy(dtype=float)
    for (u, v), c in trans_cnt.items():
        dist = haversine_km(lat_arr[u], lon_arr[u], lat_arr[v], lon_arr[v])
        add_edge(int(u), int(v), dist, "trans", c)

    # add KNN edges (bidirectional) to keep graph connected
    X = np.column_stack(_equirect_xy_km(lat_arr, lon_arr, float(np.nanmedian(lat_arr))))
    nn_idx = _try_ckdtree_knn(X, int(cfg["knn_k"]))
    for u in range(len(node_df)):
        for v in nn_idx[u]:
            v = int(v)
            if v == u:
                continue
            dist = haversine_km(lat_arr[u], lon_arr[u], lat_arr[v], lon_arr[v])
            # only add if not already a transition edge (still allow; but avoid duplicates)
            add_edge(u, v, dist, "knn", 1)
            add_edge(v, u, dist, "knn", 1)

    # attach origin/dest port nodes
    if o not in ports_map or d not in ports_map:
        return None, dict(reason="skip: port not found in ports_df", origin=o, dest=d)

    o_lat, o_lon360 = ports_map[o]
    d_lat, d_lon360 = ports_map[d]

    # find nearest nodes for port connections using KDTree on node XY
    Xn = np.column_stack(_equirect_xy_km(lat_arr, lon_arr, float(np.nanmedian(lat_arr))))
    Xo = np.array(_equirect_xy_km(np.array([o_lat]), np.array([o_lon360]), float(np.nanmedian(lat_arr)))).T
    Xd = np.array(_equirect_xy_km(np.array([d_lat]), np.array([d_lon360]), float(np.nanmedian(lat_arr)))).T

    try:
        from scipy.spatial import cKDTree
        tree = cKDTree(Xn)
        kpc = min(int(cfg["port_connect_k"]), len(Xn))
        _, idx_o = tree.query(Xo[0], k=kpc)
        _, idx_d = tree.query(Xd[0], k=kpc)
        idx_o = np.atleast_1d(idx_o).tolist()
        idx_d = np.atleast_1d(idx_d).tolist()
    except Exception:
        # fallback brute
        kpc = min(int(cfg["port_connect_k"]), len(Xn))
        dd_o = np.sum((Xn - Xo[0])**2, axis=1)
        dd_d = np.sum((Xn - Xd[0])**2, axis=1)
        idx_o = np.argsort(dd_o)[:kpc].tolist()
        idx_d = np.argsort(dd_d)[:kpc].tolist()

    # add port nodes
    O_NODE = int(len(node_df))
    D_NODE = int(len(node_df) + 1)
    adj[O_NODE] = []
    adj[D_NODE] = []

    # connect O -> nearest nodes, and nearest nodes -> D (directed)
    for v in idx_o:
        dist = haversine_km(o_lat, o_lon360, lat_arr[v], lon_arr[v])
        add_edge(O_NODE, int(v), dist, "port", 1)
    for u in idx_d:
        dist = haversine_km(lat_arr[u], lon_arr[u], d_lat, d_lon360)
        add_edge(int(u), D_NODE, dist, "port", 1)

    # run shortest path
    path_nodes, path_edges, total_cost = _dijkstra(adj, O_NODE, D_NODE)
    if path_nodes is None or len(path_nodes) < 3:
        return None, dict(reason="fail: no path O->D in corridor graph", n_nodes=len(node_df), n_voyages=len(vids))

    # build polyline (drop the two port nodes in middle; keep start/end as ports)
    seq_lat = []
    seq_lon360 = []
    seq_support = []
    seq_kind = []

    # start with origin port
    seq_lat.append(float(o_lat))
    seq_lon360.append(float(o_lon360))
    seq_support.append(0)
    seq_kind.append("port")

    # internal nodes
    for nid in path_nodes[1:-1]:
        if nid >= len(node_df):  # safety
            continue
        seq_lat.append(float(lat_arr[nid]))
        seq_lon360.append(float(lon_arr[nid]))
        seq_support.append(0)  # will fill below
        seq_kind.append("node")

    # end with dest port
    seq_lat.append(float(d_lat))
    seq_lon360.append(float(d_lon360))
    seq_support.append(0)
    seq_kind.append("port")

    # fill support/kind from edges (align roughly; we store edge meta per hop)
    # We have edges for hops: O->n1, n1->n2, ..., nk->D
    # We'll map edge meta to the "to" point index (1..)
    for i, meta in enumerate(path_edges, start=1):
        if i < len(seq_support):
            seq_support[i] = int(meta.get("support", 0))
            seq_kind[i] = meta.get("kind", seq_kind[i])

    ref_df = pd.DataFrame({
        "seq": np.arange(len(seq_lat), dtype=int),
        "Lat": seq_lat,
        "Long_360": seq_lon360,
        "Long": [float(_to_lon180(x)) for x in seq_lon360],
        "edge_support": seq_support,
        "edge_kind": seq_kind,
    })

    # compute ref length
    ref_len = 0.0
    for i in range(1, len(ref_df)):
        ref_len += haversine_km(ref_df.loc[i-1,"Lat"], ref_df.loc[i-1,"Long_360"],
                                ref_df.loc[i,"Lat"],   ref_df.loc[i,"Long_360"])

    meta = dict(
        origin_port_id=str(o),
        dest_port_id=str(d),
        n_voyages_used=int(len(vids)),
        n_points_used=int(len(g_od)),
        n_nodes=int(len(node_df)),
        n_trans_edges=int(len(trans_cnt)),
        ref_len_km=float(ref_len),
        total_cost=float(total_cost),
        ref_select_reason="model_ref_knn_corridor_shortestpath",
    )
    return ref_df, meta


# --------------------
# Run for all ODs
# --------------------
ports_map = _safe_ports_map(ports_df)

ref_polylines = {}
ref_rows = []

if df_clean_points_ref_pool.empty:
    print("[Cell2] WARNING: df_clean_points_ref_pool is empty -> no ref can be built.")
    ref_routes_df = pd.DataFrame(columns=[
        "origin_port_id","dest_port_id","n_voyages_used","ref_select_reason"
    ])
else:
    # ensure OD columns are string
    pool = df_clean_points_ref_pool.loc[:, ~df_clean_points_ref_pool.columns.duplicated()].copy()
    pool["origin_port_id"] = pool["origin_port_id"].astype(str)
    pool["dest_port_id"] = pool["dest_port_id"].astype(str)

    for (o, d), g_od in pool.groupby(["origin_port_id","dest_port_id"]):
        ref_df, meta = build_model_ref_for_one_od(g_od, str(o), str(d), ports_map, REF_A_CFG)
        if ref_df is not None and len(ref_df) >= 2:
            ref_polylines[(str(o), str(d))] = ref_df
        ref_rows.append(meta)

    ref_routes_df = pd.DataFrame(ref_rows).sort_values(["origin_port_id","dest_port_id"]).reset_index(drop=True)

print("[Cell2] Built ref ODs:", int((ref_routes_df["ref_select_reason"] == "model_ref_knn_corridor_shortestpath").sum()) if len(ref_routes_df) else 0)
print("[Cell2] ref_polylines keys:", len(ref_polylines))
display(ref_routes_df.head(20))


[Cell2] Built ref ODs: 7
[Cell2] ref_polylines keys: 7


,origin_port_id,dest_port_id,n_voyages_used,n_points_used,n_nodes,n_trans_edges,ref_len_km,total_cost,ref_select_reason,reason,n_voyages
0,CNFOC,TWTPE,20.0,6296.0,1031.0,1949.0,249.725690,94.890368,model_ref_knn_corridor_shortestpath,NaN,NaN
1,CNLYA,TWTPE,7.0,2151.0,791.0,1064.0,248.154554,164.094640,model_ref_knn_corridor_shortestpath,NaN,NaN
2,JPNGO,TWTPE,3.0,6633.0,3732.0,4115.0,1953.930425,1868.553960,model_ref_knn_corridor_shortestpath,NaN,NaN
3,TWTPE,CNFOC,19.0,5541.0,1046.0,1918.0,248.558614,103.026007,model_ref_knn_corridor_shortestpath,NaN,NaN
4,TWTPE,CNLYA,4.0,1381.0,611.0,773.0,245.904503,201.417840,model_ref_knn_corridor_shortestpath,NaN,NaN
5,TWTPE,JPNGO,2.0,3856.0,2532.0,2628.0,1950.504648,1922.134914,model_ref_knn_corridor_shortestpath,NaN,NaN
6,TWTPE,TWKEL,3.0,313.0,127.0,152.0,71.175273,56.421679,model_ref_knn_corridor_shortestpath,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,skip: n_voyages<2,1.0
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,skip: n_voyages<2,1.0
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,skip: n_voyages<2,1.0


### KNN 結果

In [23]:
# ---- pick only KNN-built refs ----
rr_knn = ref_routes_df.copy()
rr_knn["ref_select_reason"] = rr_knn.get("ref_select_reason", "").astype(str)

rr_knn = rr_knn[rr_knn["ref_select_reason"].str.contains("knn", case=False, na=False)].copy()

if rr_knn.empty:
    print("[WARN] ref_routes_df 裡找不到 knn 的 ref_select_reason，檢查一下你 step5 是否真的跑的是 KNN corridor 版本。")
else:
    # choose top ODs by n_voyages_used (or just head)
    if "n_voyages_used" in rr_knn.columns:
        rr_knn = rr_knn.sort_values("n_voyages_used", ascending=False)

    od_list = list(rr_knn[["origin_port_id","dest_port_id"]].head(30).itertuples(index=False, name=None))

    visualize_model_ref_routes_by_od(
        ref_routes_df=rr_knn,
        ref_polylines=ref_polylines,
        ports_df=ports_df,
        od_list=od_list,                 # 你也可以只放一組 [('CNFOC','TWTPE')]
        output_html="outputs/knn_model_ref_routes_map.html",
        zoom_start=6,
        add_port_markers=True,
        line_weight=5,
        line_opacity=0.9,
        show_coverage=False,             # KNN route 通常沒有 coverage_n
    )


已輸出地圖到 C:\Users\slab\Desktop\Slab Project\Stage2 ETA\outputs\knn_model_ref_routes_map.html


In [24]:
import folium
from folium import LayerControl
from branca.element import Element
import pandas as pd
import numpy as np

def visualize_knn_ref_route_segmented(
    ref_routes_df: pd.DataFrame,
    ref_polylines: dict,
    ports_df: pd.DataFrame | None = None,
    od_list=None,
    max_od: int = 20,
    output_html="outputs/knn_ref_segmented_map.html",
    zoom_start=6,
):
    # ---- choose ODs ----
    rr = ref_routes_df.copy()
    rr["origin_port_id"] = rr["origin_port_id"].astype(str)
    rr["dest_port_id"]   = rr["dest_port_id"].astype(str)

    if od_list is None:
        rr = rr.head(max_od).copy()
        od_list = list(rr[["origin_port_id","dest_port_id"]].itertuples(index=False, name=None))
    else:
        od_list = [(str(a), str(b)) for a,b in od_list]

    # ---- ports map ----
    port_map = {}
    if ports_df is not None and isinstance(ports_df, pd.DataFrame) and not ports_df.empty:
        p = ports_df.copy()
        if "port_id" not in p.columns and "unlocode" in p.columns:
            p = p.rename(columns={"unlocode":"port_id"})
        if "port_name" not in p.columns:
            p["port_name"] = p.get("name", p.get("port_id", "PORT"))
        need = ["port_id", "lat", "lon", "port_name"]
        if all(c in p.columns for c in need):
            pp = p[need].dropna(subset=["lat","lon"]).copy()
            for _, r in pp.iterrows():
                port_map[str(r["port_id"])] = (float(r["lat"]), float(r["lon"]), str(r["port_name"]))

    # ---- map center ----
    first_key = None
    for k in od_list:
        if k in ref_polylines and len(ref_polylines[k]) >= 2:
            first_key = k
            break
    if first_key is None:
        print("[WARN] 沒有可畫的 ref polyline。")
        return None

    seg0 = ref_polylines[first_key].copy()
    seg0["Lat"]  = pd.to_numeric(seg0["Lat"], errors="coerce")
    seg0["Long"] = pd.to_numeric(seg0["Long"], errors="coerce")
    seg0 = seg0.dropna(subset=["Lat","Long"])
    if len(seg0) < 2:
        print("[WARN] 第一條 ref 清完後不足兩點。")
        return None

    m = folium.Map(location=[float(seg0["Lat"].median()), float(seg0["Long"].median())],
                   zoom_start=zoom_start, tiles="OpenStreetMap")

    # colors by edge_kind
    COLOR = {
        "trans": "blue",     # 主要歷史走廊
        "knn": "orange",     # KNN 補洞
        "port": "green",     # port connection
        "node": "blue",
        "unknown": "purple",
    }

    for (o, d) in od_list:
        key = (o, d)
        if key not in ref_polylines:
            continue

        seg = ref_polylines[key].copy()
        seg["Lat"]  = pd.to_numeric(seg["Lat"], errors="coerce")
        seg["Long"] = pd.to_numeric(seg["Long"], errors="coerce")
        seg = seg.dropna(subset=["Lat","Long"])
        if len(seg) < 2:
            continue

        # meta for layer name
        row = rr[(rr["origin_port_id"] == o) & (rr["dest_port_id"] == d)]
        n_used = None
        reason = ""
        if not row.empty:
            n_used = row.iloc[0].get("n_voyages_used", None)
            reason = str(row.iloc[0].get("ref_select_reason", "")).strip()

        layer_parts = [f"KNN_REF {o} → {d}"]
        if pd.notna(n_used):
            layer_parts.append(f"n={int(n_used)}")
        if reason:
            layer_parts.append(reason)
        layer_name = " | ".join(layer_parts)

        fg = folium.FeatureGroup(name=layer_name)

        # segment-by-segment draw
        has_kind = "edge_kind" in seg.columns
        has_sup  = "edge_support" in seg.columns

        coords = list(zip(seg["Lat"].astype(float), seg["Long"].astype(float)))

        # Draw as segments so each hop can have different color
        for i in range(1, len(coords)):
            kind = "unknown"
            sup = None
            if has_kind:
                kind = str(seg["edge_kind"].iloc[i]).lower().strip()
            if has_sup:
                try:
                    sup = int(seg["edge_support"].iloc[i])
                except Exception:
                    sup = None

            color = COLOR.get(kind, COLOR["unknown"])
            weight = 6 if kind == "trans" else (5 if kind == "knn" else 4)

            tooltip = f"kind={kind}"
            if sup is not None:
                tooltip += f", support={sup}"

            folium.PolyLine([coords[i-1], coords[i]], color=color, weight=weight, opacity=0.95,
                            tooltip=tooltip).add_to(fg)

        # start/end markers
        folium.CircleMarker(coords[0], radius=5, color="green", fill=True, fill_opacity=0.9,
                            tooltip=f"start {o}").add_to(fg)
        folium.CircleMarker(coords[-1], radius=5, color="black", fill=True, fill_opacity=0.9,
                            tooltip=f"end {d}").add_to(fg)

        # port markers
        if o in port_map:
            plat, plon, pname = port_map[o]
            folium.Marker([plat, plon], popup=f"Origin {o} ({pname})",
                          icon=folium.Icon(color="green", icon="ship", prefix="fa")).add_to(fg)
        if d in port_map:
            plat, plon, pname = port_map[d]
            folium.Marker([plat, plon], popup=f"Dest {d} ({pname})",
                          icon=folium.Icon(color="darkred", icon="anchor", prefix="fa")).add_to(fg)

        fg.add_to(m)

    # layer control + select all/none
    LayerControl(collapsed=False).add_to(m)
    m.get_root().html.add_child(Element("""
    <script>
    (function() {
      function setAllOverlays(checked){
        var inputs = document.querySelectorAll(
          '.leaflet-control-layers-overlays input[type="checkbox"]'
        );
        inputs.forEach(function(input){
          if (input.checked !== checked){ input.click(); }
        });
      }
      function injectButtons(){
        var lc = document.querySelector('.leaflet-control-layers');
        if(!lc) return;
        if (lc.querySelector('.folium-layer-toggle-all')) return;
        var form = lc.querySelector('form') || lc;
        var box = document.createElement('div');
        box.className = 'folium-layer-toggle-all';
        box.style.padding = '6px 8px';
        box.style.borderBottom = '1px solid #ddd';
        box.style.display = 'flex';
        box.style.gap = '6px';

        function mkBtn(txt){
          var b = document.createElement('button');
          b.type = 'button';
          b.textContent = txt;
          b.style.cursor = 'pointer';
          return b;
        }
        var btnAll = mkBtn('全選');
        var btnNone = mkBtn('全不選');
        btnAll.onclick = function(){ setAllOverlays(true); };
        btnNone.onclick = function(){ setAllOverlays(false); };
        box.appendChild(btnAll);
        box.appendChild(btnNone);
        form.insertBefore(box, form.firstChild);
      }
      window.addEventListener('load', injectButtons);
    })();
    </script>
    """))

    from pathlib import Path
    outp = Path(output_html)
    outp.parent.mkdir(parents=True, exist_ok=True)
    m.save(str(outp))
    print(f"已輸出地圖到 {outp.resolve()}")

    try:
        import webbrowser
        webbrowser.open(outp.resolve().as_uri())
    except Exception:
        pass

    return m


# ---- example call ----
# 你可以指定某個 OD
# visualize_knn_ref_route_segmented(ref_routes_df, ref_polylines, ports_df, od_list=[("CNFOC","TWTPE")])

# 或者：只看 ref_select_reason 有 knn 的前 20 組
rr_knn = ref_routes_df.copy()
rr_knn["ref_select_reason"] = rr_knn.get("ref_select_reason", "").astype(str)
rr_knn = rr_knn[rr_knn["ref_select_reason"].str.contains("knn", case=False, na=False)].copy()
rr_knn = rr_knn.sort_values("n_voyages_used", ascending=False) if "n_voyages_used" in rr_knn.columns else rr_knn

od_list = list(rr_knn[["origin_port_id","dest_port_id"]].head(20).itertuples(index=False, name=None))
visualize_knn_ref_route_segmented(rr_knn, ref_polylines, ports_df, od_list=od_list,
                                  output_html="outputs/knn_ref_segmented_map.html")


已輸出地圖到 C:\Users\slab\Desktop\Slab Project\Stage2 ETA\outputs\knn_ref_segmented_map.html


## KNN baseline 作法

### import

In [20]:
import numpy as np
import pandas as pd

from pathlib import Path

# --- 你跑完上面 pipeline 後，應該有 df / ports_df / stops_labeled ---
# df: teleport-cleaned AIS points (single MMSI), has Timestamp, Lat, Long, Sog...
# ports_df: port list with port_id, lat, lon
# stops_labeled: stop episodes table, has stop_type, t_start, t_end, waiting_for_dest_port_id ...

# --- 選擇你要採用的 voyages 表（優先：已整合 loitering summary / edge-guard / large_time_gap） ---
if "voyages_with_waiting_merged" in globals():
    voyages_base = voyages_with_waiting_merged.copy()
elif "voyages_with_waiting_edgeguarded_3pts" in globals():
    voyages_base = voyages_with_waiting_edgeguarded_3pts.copy()
elif "voyages_with_waiting_enhanced" in globals():
    voyages_base = voyages_with_waiting_enhanced.copy()
else:
    voyages_base = voyages_with_waiting.copy()

# --- 選擇你要用的 loitering events（優先：merge 後；其次 kept_3pts；最後原始） ---
if "loitering_events_merged" in globals():
    loiter_events_base = loitering_events_merged.copy()
elif "loitering_events_kept_3pts" in globals():
    loiter_events_base = loitering_events_kept_3pts.copy()
elif "loitering_events" in globals():
    loiter_events_base = loitering_events.copy()
else:
    loiter_events_base = pd.DataFrame(columns=["voyage_id","t_start","t_end"])

# --- 基本欄位確保 datetime ---
df = df.copy()
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
voyages_base = voyages_base.copy()
voyages_base["t_depart_origin"] = pd.to_datetime(voyages_base["t_depart_origin"], errors="coerce")
voyages_base["t_arrive_dest"]   = pd.to_datetime(voyages_base["t_arrive_dest"], errors="coerce")

stops_labeled = stops_labeled.copy()
if "t_start" in stops_labeled.columns:
    stops_labeled["t_start"] = pd.to_datetime(stops_labeled["t_start"], errors="coerce")
if "t_end" in stops_labeled.columns:
    stops_labeled["t_end"]   = pd.to_datetime(stops_labeled["t_end"], errors="coerce")

loiter_events_base = loiter_events_base.copy()
for c in ["t_start","t_end"]:
    if c in loiter_events_base.columns:
        loiter_events_base[c] = pd.to_datetime(loiter_events_base[c], errors="coerce")

print("voyages_base:", len(voyages_base))
print("loiter_events_base:", len(loiter_events_base))
print("df points:", len(df))


voyages_base: 99
loiter_events_base: 114
df points: 200464


### 距離、區間遮罩（把 anchorage/loitering 變成 point mask）

In [21]:
from math import radians, sin, cos, atan2, sqrt

def _to_lon180(dlon):
    return ((dlon + 180.0) % 360.0) - 180.0

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    p1, p2 = radians(lat1), radians(lat2)
    dphi = radians(lat2 - lat1)
    dlmb = radians(_to_lon180(lon2 - lon1))
    a = sin(dphi/2)**2 + cos(p1)*cos(p2)*sin(dlmb/2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1-a))

def polyline_length_km(lat, lon):
    lat = np.asarray(lat, float)
    lon = np.asarray(lon, float)
    if len(lat) < 2:
        return 0.0
    tot = 0.0
    for i in range(1, len(lat)):
        if np.isnan(lat[i-1]) or np.isnan(lon[i-1]) or np.isnan(lat[i]) or np.isnan(lon[i]):
            continue
        tot += haversine_km(lat[i-1], lon[i-1], lat[i], lon[i])
    return float(tot)

def mask_times_in_intervals(times: pd.Series, intervals: list[tuple[pd.Timestamp, pd.Timestamp]]) -> np.ndarray:
    """
    times: sorted datetime Series
    intervals: list of (start, end)
    return: boolean mask, True if time is within any interval
    """
    if times.empty or not intervals:
        return np.zeros(len(times), dtype=bool)

    t = times.to_numpy()
    intervals = [(pd.to_datetime(a), pd.to_datetime(b)) for a, b in intervals if pd.notna(a) and pd.notna(b) and b >= a]
    if not intervals:
        return np.zeros(len(times), dtype=bool)

    intervals = sorted(intervals, key=lambda x: x[0])
    mask = np.zeros(len(t), dtype=bool)

    j = 0
    cur_a, cur_b = intervals[0]
    for i in range(len(t)):
        ti = t[i]
        while j < len(intervals) and ti > cur_b:
            j += 1
            if j < len(intervals):
                cur_a, cur_b = intervals[j]
        if j >= len(intervals):
            break
        if (ti >= cur_a) and (ti <= cur_b):
            mask[i] = True
    return mask


### 建立「clean sailing 點」(去掉 large_time_gap 航程 + 去掉 anchorage/loitering 區間)

In [22]:
# ============================================================
# Clean sailing points builder (fixed):
# - Drop large_time_gap voyages
# - Remove anchorage_like intervals (full)
# - Remove loitering intervals (per-voyage)
# - Remove ONLY port_call CORE (keep edges on both sides)
#
# Inputs:
#   df                 (teleport-cleaned points, includes Timestamp/Lat/Long)
#   voyages_base        (has voyage_id, origin_port_id, dest_port_id, t_depart_origin, t_arrive_dest, large_time_gap_flag)
#   stops_labeled       (has stop_type, t_start, t_end)  <-- includes port_call, anchorage_like
#   loiter_events_base  (has voyage_id, t_start, t_end)  <-- loitering events (merged/edge-guarded)
#
# Outputs:
#   df_points_labeled   (all voyage points with bad_flag/clean_flag)
#   df_clean_points_all (only clean_flag points)
# ============================================================

import numpy as np
import pandas as pd

# --------------------------
# Config
# --------------------------
CFG_CLEAN = dict(
    drop_large_time_gap=True,
    drop_anchorage_like=True,
    drop_loitering=True,

    # ★ 핵심修正：port_call 不整段砍掉，只砍核心，保留兩側出/進港航行點
    drop_port_call_core=True,
    portcall_keep_edge_min=20,   # 保留兩側 20 分鐘（可調 10/20/30）
)

# --------------------------
# Helpers
# --------------------------
def merge_intervals(intervals):
    """intervals: list[(start,end)] -> merged, sorted, non-overlap"""
    if not intervals:
        return []
    itv = sorted(intervals, key=lambda x: x[0])
    out = [[itv[0][0], itv[0][1]]]
    for s, e in itv[1:]:
        if s <= out[-1][1]:
            out[-1][1] = max(out[-1][1], e)
        else:
            out.append([s, e])
    return [(a, b) for a, b in out]

def clip_intervals_to_window(intervals, t0, t1):
    """clip to [t0,t1] and drop invalid; then merge"""
    out = []
    for a, b in intervals:
        aa = max(a, t0)
        bb = min(b, t1)
        if bb > aa:
            out.append((aa, bb))
    return merge_intervals(out)

def mask_times_in_intervals(ts: pd.Series, intervals):
    """True if timestamp falls in ANY interval (inclusive)."""
    if ts is None or ts.empty:
        return np.zeros(0, dtype=bool)
    if not intervals:
        return np.zeros(len(ts), dtype=bool)
    t = ts.to_numpy(dtype="datetime64[ns]")
    bad = np.zeros(len(t), dtype=bool)
    for a, b in intervals:
        a64 = np.datetime64(a)
        b64 = np.datetime64(b)
        bad |= (t >= a64) & (t <= b64)
    return bad

# --------------------------
# 0) Normalize inputs
# --------------------------
voyages_use = voyages_base.copy()
voyages_use["origin_port_id"] = voyages_use["origin_port_id"].astype(str)
voyages_use["dest_port_id"]   = voyages_use["dest_port_id"].astype(str)

voyages_use["t_depart_origin"] = pd.to_datetime(voyages_use["t_depart_origin"], errors="coerce")
voyages_use["t_arrive_dest"]   = pd.to_datetime(voyages_use["t_arrive_dest"], errors="coerce")

if CFG_CLEAN["drop_large_time_gap"] and ("large_time_gap_flag" in voyages_use.columns):
    voyages_use = voyages_use[~voyages_use["large_time_gap_flag"].fillna(False)].copy()

voyages_use = voyages_use.dropna(subset=["t_depart_origin", "t_arrive_dest"]).copy()
voyages_use = voyages_use[voyages_use["t_arrive_dest"] > voyages_use["t_depart_origin"]].copy()
voyages_use = voyages_use.sort_values("t_depart_origin").reset_index(drop=True)

print("voyages_use:", len(voyages_use))

# points
df_sorted = df.copy()
df_sorted["Timestamp"] = pd.to_datetime(df_sorted["Timestamp"], errors="coerce")
df_sorted["Lat"] = pd.to_numeric(df_sorted["Lat"], errors="coerce")
df_sorted["Long"] = pd.to_numeric(df_sorted["Long"], errors="coerce")
df_sorted = df_sorted.sort_values("Timestamp").dropna(subset=["Timestamp","Lat","Long"]).copy()

# stops -> anch / port
stops_ok = stops_labeled.copy() if isinstance(stops_labeled, pd.DataFrame) and (not stops_labeled.empty) else pd.DataFrame()
if not stops_ok.empty:
    stops_ok["t_start"] = pd.to_datetime(stops_ok["t_start"], errors="coerce")
    stops_ok["t_end"]   = pd.to_datetime(stops_ok["t_end"], errors="coerce")
    stops_ok["stop_type"] = stops_ok["stop_type"].astype(str).str.lower()
    stops_ok = stops_ok.dropna(subset=["t_start","t_end"])
    stops_ok = stops_ok[stops_ok["t_end"] > stops_ok["t_start"]].copy()

anch = stops_ok[stops_ok["stop_type"].isin(["anchorage_like","anchored_like","anchored"])].copy() if CFG_CLEAN["drop_anchorage_like"] and (not stops_ok.empty) else pd.DataFrame(columns=["t_start","t_end"])
port = stops_ok[stops_ok["stop_type"].isin(["port_call"])].copy() if CFG_CLEAN["drop_port_call_core"] and (not stops_ok.empty) else pd.DataFrame(columns=["t_start","t_end"])

# loiter events
loit = loiter_events_base.copy() if isinstance(loiter_events_base, pd.DataFrame) and (not loiter_events_base.empty) else pd.DataFrame(columns=["voyage_id","t_start","t_end"])
if CFG_CLEAN["drop_loitering"] and (not loit.empty):
    loit["t_start"] = pd.to_datetime(loit["t_start"], errors="coerce")
    loit["t_end"]   = pd.to_datetime(loit["t_end"], errors="coerce")
    if "voyage_id" in loit.columns:
        loit["voyage_id"] = pd.to_numeric(loit["voyage_id"], errors="coerce").astype("Int64")
    loit = loit.dropna(subset=["t_start","t_end"])
    loit = loit[loit["t_end"] > loit["t_start"]].copy()
else:
    loit = pd.DataFrame(columns=["voyage_id","t_start","t_end"])

keep_edge = pd.Timedelta(minutes=float(CFG_CLEAN["portcall_keep_edge_min"]))

# --------------------------
# 1) Per-voyage cut & clean mask
# --------------------------
rows_all = []

for _, v in voyages_use.iterrows():
    vid = int(v["voyage_id"])
    t0, t1 = v["t_depart_origin"], v["t_arrive_dest"]

    seg = df_sorted[(df_sorted["Timestamp"] >= t0) & (df_sorted["Timestamp"] <= t1)].copy()
    if len(seg) < 2:
        continue

    seg = seg.sort_values("Timestamp").reset_index(drop=False).rename(columns={"index":"orig_index"})
    seg["voyage_id"] = vid
    seg["origin_port_id"] = str(v["origin_port_id"])
    seg["dest_port_id"] = str(v["dest_port_id"])

    intervals = []

    # (A) anchorage_like: full interval remove (overlap voyage window)
    if not anch.empty:
        hit = anch[(anch["t_start"] <= t1) & (anch["t_end"] >= t0)]
        if not hit.empty:
            intervals += [(r["t_start"], r["t_end"]) for _, r in hit.iterrows()]

    # (B) loitering: per-voyage intervals remove
    if not loit.empty:
        if "voyage_id" in loit.columns:
            hit = loit[loit["voyage_id"] == vid]
        else:
            hit = loit
        hit = hit[(hit["t_start"] <= t1) & (hit["t_end"] >= t0)]
        if not hit.empty:
            intervals += [(r["t_start"], r["t_end"]) for _, r in hit.iterrows()]

    # (C) port_call: ★ core-shrink (keep edges)
    #     Only remove [t_start+edge, t_end-edge] if it is valid (end > start)
    if not port.empty and CFG_CLEAN["portcall_keep_edge_min"] > 0:
        hit = port[(port["t_start"] <= t1) & (port["t_end"] >= t0)]
        for _, r in hit.iterrows():
            a = r["t_start"] + keep_edge
            b = r["t_end"]   - keep_edge
            if pd.notna(a) and pd.notna(b) and (b > a):
                intervals.append((a, b))
            # else: port_call too short -> keep whole interval (do nothing)

    # merge + clip to voyage window
    intervals = clip_intervals_to_window(merge_intervals(intervals), t0, t1)

    bad = mask_times_in_intervals(seg["Timestamp"], intervals)
    seg["bad_flag"] = bad
    seg["clean_flag"] = ~bad

    rows_all.append(seg)

df_points_labeled = pd.concat(rows_all, ignore_index=True) if rows_all else pd.DataFrame()
print("df_points_labeled:", df_points_labeled.shape)

df_clean_points_all = df_points_labeled[df_points_labeled["clean_flag"]].copy().reset_index(drop=True)
print("df_clean_points_all:", df_clean_points_all.shape)

display(df_clean_points_all.head(3))


voyages_use: 92
df_points_labeled: (67898, 61)
df_clean_points_all: (50389, 61)


,orig_index,orig_index,orig_index_old,orig_index_old,orig_index_old,orig_index_old,orig_index_old,orig_index_old,orig_index_old,Pky,...,B_Depth,B_WindSpeed,B_WindAngle,B_RPM,Long_360,voyage_id,origin_port_id,dest_port_id,bad_flag,clean_flag
0,296,296,296,296,296,296,296,296,296,261088,...,NaN,NaN,NaN,NaN,119.480165,1,CNFOC,TWTPE,False,True
1,297,297,297,297,297,297,297,297,297,261089,...,NaN,NaN,NaN,NaN,119.481100,1,CNFOC,TWTPE,False,True
2,298,298,298,298,298,298,298,298,298,261090,...,NaN,NaN,NaN,NaN,119.480722,1,CNFOC,TWTPE,False,True


### OD level, frop 10% voyages

In [23]:
CFG_POOL = dict(
    trim_pct=0.10,        # OD-wise 丟最長/最短 10% 航程
    min_points_voy=30,    # 每條航程至少要多少 clean 點才有意義
    min_voyages_od=3,     # 一個 OD 至少幾條航程才做 baseline/ref (太少不穩)
)

# 每條 voyage 的 clean distance
def compute_voyage_clean_km(seg: pd.DataFrame) -> float:
    seg = seg.sort_values("Timestamp")
    lat = pd.to_numeric(seg["Lat"], errors="coerce").to_numpy(float)
    lon = pd.to_numeric(seg["Long"], errors="coerce").to_numpy(float)
    return polyline_length_km(lat, lon)

voy_metrics = []
for vid, g in df_clean_points_all.groupby("voyage_id"):
    g = g.sort_values("Timestamp")
    if len(g) < 2:
        continue
    voy_metrics.append({
        "voyage_id": int(vid),
        "origin_port_id": str(g["origin_port_id"].iloc[0]),
        "dest_port_id": str(g["dest_port_id"].iloc[0]),
        "n_clean_points": int(len(g)),
        "km_clean": float(compute_voyage_clean_km(g)),
        "t_start_clean": g["Timestamp"].iloc[0],
        "t_end_clean": g["Timestamp"].iloc[-1],
    })
voy_metrics = pd.DataFrame(voy_metrics)
print("voy_metrics:", len(voy_metrics))

# 先做 min_points 過濾
voy_metrics = voy_metrics[voy_metrics["n_clean_points"] >= CFG_POOL["min_points_voy"]].copy()

# OD-wise trim（丟最長/最短 10%）
keep_vids = []
for (o, d), g in voy_metrics.groupby(["origin_port_id","dest_port_id"]):
    g = g.sort_values("km_clean").reset_index(drop=True)
    n = len(g)
    if n < CFG_POOL["min_voyages_od"]:
        continue
    k = int(np.floor(CFG_POOL["trim_pct"] * n))
    lo = k
    hi = n - k
    gk = g.iloc[lo:hi] if hi > lo else g.iloc[0:0]
    keep_vids.extend(gk["voyage_id"].tolist())

keep_vids = sorted(set(keep_vids))
print("kept voyages after OD-trim:", len(keep_vids))

# ref/baseline pool points
df_clean_points_pool = df_clean_points_all[df_clean_points_all["voyage_id"].isin(keep_vids)].copy()
print("df_clean_points_pool:", df_clean_points_pool.shape)

# OD 統計一下
od_stats = df_clean_points_pool.groupby(["origin_port_id","dest_port_id"])["voyage_id"].nunique().reset_index(name="n_voyages")
display(od_stats.sort_values("n_voyages", ascending=False).head(10))


voy_metrics: 92
kept voyages after OD-trim: 64
df_clean_points_pool: (24988, 61)


,origin_port_id,dest_port_id,n_voyages
0,CNFOC,TWTPE,22
3,TWTPE,CNFOC,21
1,CNLYA,TWTPE,8
4,TWTPE,CNLYA,7
2,JPNGO,TWTPE,3
5,TWTPE,TWKEL,3


### KNN baseline

In [24]:
from sklearn.neighbors import NearestNeighbors
import numpy as np
import pandas as pd
import heapq
from math import radians, cos

CFG_BASELINE = dict(
    grid_deg=0.05,
    min_bin_count=8,       # 你可先不改；如果港口附近常被濾掉，再降到 3 或 1
    k_nn=10,
    dens_alpha=0.7,

    # ★新增：港口節點連接設定（模仿你舊版「port_connect_k + port_penalty」精神）
    port_connect_k=6,
    port_penalty=2.0,
)

def _xy_km_from_latlon(lat, lon, lat0):
    """
    簡單 local equirect：讓 KNN 用 km 空間找鄰居（比直接用 degree 可靠很多）
    """
    lat = np.asarray(lat, float)
    lon = np.asarray(lon, float)
    x = lon * cos(radians(lat0)) * 111.320
    y = lat * 110.574
    return np.column_stack([x, y])

def build_density_nodes(points_od: pd.DataFrame, grid_deg=0.05, min_bin_count=8):
    g = points_od.copy()
    g["Lat"] = pd.to_numeric(g["Lat"], errors="coerce")
    g["Long"] = pd.to_numeric(g["Long"], errors="coerce")
    g = g.dropna(subset=["Lat","Long"])
    if g.empty:
        return pd.DataFrame(columns=["node_id","lat","lon","count","is_port"])

    lat_bin = np.floor(g["Lat"] / grid_deg).astype(int)
    lon_bin = np.floor(g["Long"] / grid_deg).astype(int)

    agg = g.groupby([lat_bin, lon_bin]).size().reset_index(name="count")
    agg = agg[agg["count"] >= min_bin_count].copy()
    if agg.empty:
        return pd.DataFrame(columns=["node_id","lat","lon","count","is_port"])

    agg["lat"] = (agg.iloc[:,0] + 0.5) * grid_deg
    agg["lon"] = (agg.iloc[:,1] + 0.5) * grid_deg
    agg = agg.reset_index(drop=True)
    agg["node_id"] = np.arange(len(agg), dtype=int)
    agg["is_port"] = False
    return agg[["node_id","lat","lon","count","is_port"]]

def build_knn_graph_km(nodes_df: pd.DataFrame, k=10):
    """
    用 km 平面做 KNN，回傳 neigh[u] = [(v, dist_km), ...]
    """
    if nodes_df.empty or len(nodes_df) < 2:
        return [[] for _ in range(len(nodes_df))]

    lat0 = float(np.nanmedian(nodes_df["lat"].to_numpy(float)))
    X = _xy_km_from_latlon(nodes_df["lat"].to_numpy(float), nodes_df["lon"].to_numpy(float), lat0)

    nn = NearestNeighbors(n_neighbors=min(k+1, len(nodes_df)), algorithm="auto").fit(X)
    dists, idxs = nn.kneighbors(X)   # dists 已經是 km

    neigh = [[] for _ in range(len(nodes_df))]
    for i in range(len(nodes_df)):
        for dd, j in zip(dists[i][1:], idxs[i][1:]):  # skip itself
            neigh[i].append((int(j), float(dd)))
    return neigh

def astar_density_path(nodes_df, neigh, s, t, dens_alpha=0.7, port_penalty=2.0):
    """
    A*：一般節點用 density cost；只要碰到港口節點就用 port_penalty * dist_km
    """
    if nodes_df.empty:
        return None

    latn = nodes_df["lat"].to_numpy(float)
    lonn = nodes_df["lon"].to_numpy(float)
    dens = np.maximum(nodes_df["count"].to_numpy(float), 1.0)
    is_port = nodes_df["is_port"].to_numpy(bool)

    def _h(i):
        return haversine_km(latn[i], lonn[i], latn[t], lonn[t])

    open_heap = []
    heapq.heappush(open_heap, (0.0, int(s)))
    came = {int(s): None}
    gscore = {int(s): 0.0}

    while open_heap:
        _, cur = heapq.heappop(open_heap)
        cur = int(cur)
        if cur == int(t):
            path = []
            x = cur
            while x is not None:
                path.append(int(x))
                x = came.get(x, None)
            path.reverse()
            return path

        for nxt, dist_km in neigh[cur]:
            nxt = int(nxt)

            # ★成本：港口連接邊用懲罰；一般邊用 density reward
            if is_port[cur] or is_port[nxt]:
                w = dist_km * float(port_penalty)
            else:
                w = dist_km / ((dens[cur] * dens[nxt]) ** float(dens_alpha))

            cand = gscore[cur] + w
            if (nxt not in gscore) or (cand < gscore[nxt]):
                gscore[nxt] = cand
                came[nxt] = cur
                f = cand + _h(nxt)
                heapq.heappush(open_heap, (f, nxt))

    return None

def add_port_nodes(nodes_df, o_latlon, d_latlon):
    """
    把港口當成「明確節點」加入圖中，保證路徑從港口出發/抵達
    """
    nodes = nodes_df.copy().reset_index(drop=True)
    next_id = int(nodes["node_id"].max()+1) if len(nodes) else 0

    o_row = {"node_id": next_id,   "lat": float(o_latlon[0]), "lon": float(o_latlon[1]), "count": 1.0, "is_port": True}
    d_row = {"node_id": next_id+1, "lat": float(d_latlon[0]), "lon": float(d_latlon[1]), "count": 1.0, "is_port": True}

    nodes = pd.concat([nodes, pd.DataFrame([o_row, d_row])], ignore_index=True)
    # 保證 node_id 對 index 一致（最簡單：直接用 index 當 node_id）
    nodes["node_id"] = np.arange(len(nodes), dtype=int)
    return nodes, int(len(nodes)-2), int(len(nodes)-1)  # (nodes, O_ID, D_ID)

# ---- build baseline per OD ----
port_map = {str(r["port_id"]):(float(r["lat"]), float(r["lon"]))
            for _, r in ports_df[["port_id","lat","lon"]].dropna().iterrows()}

baselines_by_od = {}
baseline_meta = []

for (o, d), g in df_clean_points_pool.groupby(["origin_port_id","dest_port_id"]):
    o = str(o); d = str(d)
    n_v = g["voyage_id"].nunique()
    if n_v < CFG_POOL["min_voyages_od"]:
        continue
    if (o not in port_map) or (d not in port_map):
        continue

    nodes0 = build_density_nodes(g, grid_deg=CFG_BASELINE["grid_deg"], min_bin_count=CFG_BASELINE["min_bin_count"])
    if len(nodes0) < 2:
        baseline_meta.append({"origin_port_id":o,"dest_port_id":d,"status":"fail_nodes_too_few","n_v":int(n_v)})
        continue

    # ★關鍵：加入 O/D 港口節點，並把它們納入 KNN 圖
    nodes, O_ID, D_ID = add_port_nodes(nodes0, port_map[o], port_map[d])

    neigh = build_knn_graph_km(nodes, k=CFG_BASELINE["k_nn"])

    # A* from O_ID to D_ID
    path_idx = astar_density_path(
        nodes, neigh, O_ID, D_ID,
        dens_alpha=CFG_BASELINE["dens_alpha"],
        port_penalty=CFG_BASELINE["port_penalty"],
    )

    if not path_idx or len(path_idx) < 2:
        baseline_meta.append({"origin_port_id":o,"dest_port_id":d,"status":"fail_astar","n_v":int(n_v), "n_nodes":int(len(nodes))})
        continue

    lat = nodes.loc[path_idx, "lat"].to_numpy(float)
    lon = nodes.loc[path_idx, "lon"].to_numpy(float)

    step_km = [0.0]
    for i in range(1, len(lat)):
        step_km.append(haversine_km(lat[i-1], lon[i-1], lat[i], lon[i]))
    step_km = np.array(step_km, float)
    cum_km = np.cumsum(step_km)
    total = float(cum_km[-1]) if len(cum_km) else 0.0
    p = cum_km / total if total > 0 else np.zeros_like(cum_km)

    base_df = pd.DataFrame({"Lat":lat, "Long":lon, "step_km":step_km, "cum_km":cum_km, "p":p})
    baselines_by_od[(o, d)] = base_df

    baseline_meta.append({"origin_port_id":o,"dest_port_id":d,"status":"ok","n_v":int(n_v), "n_nodes":int(len(nodes)), "n_path":int(len(base_df)), "base_km":total})

baseline_meta = pd.DataFrame(baseline_meta).sort_values(["status","origin_port_id","dest_port_id"]).reset_index(drop=True)
print("baselines built:", len(baselines_by_od))
display(baseline_meta.head(10))


baselines built: 6


,origin_port_id,dest_port_id,status,n_v,n_nodes,n_path,base_km
0,CNFOC,TWTPE,ok,22,84,19,238.589844
1,CNLYA,TWTPE,ok,8,77,18,243.588416
2,JPNGO,TWTPE,ok,3,368,77,1952.684150
3,TWTPE,CNFOC,ok,21,79,18,233.502062
4,TWTPE,CNLYA,ok,7,80,17,243.696940
5,TWTPE,TWKEL,ok,3,19,6,62.683920


### clean 點「投影到 baseline

In [25]:
from shapely.geometry import LineString, Point
from pyproj import Transformer

CFG_PROJECT = dict(
    min_coverage=0.80,    # clean點覆蓋 baseline 的比例（p_max - p_min）至少多少才當作 ref pool
    min_points=50,        # 投影後有效點數
)

def make_local_projector(lat0, lon0):
    """
    用 local AEQD 投影：在該 OD 區域做線性化（距離/投影較穩定）
    """
    crs_from = "EPSG:4326"
    crs_to = f"+proj=aeqd +lat_0={lat0:.6f} +lon_0={lon0:.6f} +datum=WGS84 +units=m +no_defs"
    tf = Transformer.from_crs(crs_from, crs_to, always_xy=True)
    return tf

def project_points_to_polyline(lat, lon, line_ll: pd.DataFrame):
    """
    lat/lon arrays -> project to line: along_m, xtrack_m, p
    return along_m, xtrack_m, p
    """
    # center for projection
    lat0 = float(np.nanmedian(line_ll["Lat"].to_numpy(float)))
    lon0 = float(np.nanmedian(line_ll["Long"].to_numpy(float)))
    tf = make_local_projector(lat0, lon0)

    # line
    xs, ys = tf.transform(line_ll["Long"].to_numpy(float), line_ll["Lat"].to_numpy(float))
    line = LineString(list(zip(xs, ys)))
    total_m = float(line.length) if line.length > 0 else 0.0

    along = np.full(len(lat), np.nan, float)
    xtrk  = np.full(len(lat), np.nan, float)
    p     = np.full(len(lat), np.nan, float)

    xpt, ypt = tf.transform(lon.astype(float), lat.astype(float))
    for i in range(len(lat)):
        if np.isnan(xpt[i]) or np.isnan(ypt[i]) or total_m <= 0:
            continue
        pt = Point(float(xpt[i]), float(ypt[i]))
        s = float(line.project(pt))          # along distance
        d = float(line.distance(pt))         # cross-track distance
        along[i] = s
        xtrk[i] = d
        p[i] = s / total_m

    return along, xtrk, p, total_m

# ---- project all clean points onto baseline ----
proj_rows = []
voy_cov_rows = []

for (o, d), g in df_clean_points_pool.groupby(["origin_port_id","dest_port_id"]):
    key = (str(o), str(d))
    if key not in baselines_by_od:
        continue
    base_df = baselines_by_od[key]

    for vid, gv in g.groupby("voyage_id"):
        gv = gv.sort_values("Timestamp").copy()
        lat = pd.to_numeric(gv["Lat"], errors="coerce").to_numpy(float)
        lon = pd.to_numeric(gv["Long"], errors="coerce").to_numpy(float)

        along_m, xtrk_m, p_base, base_total_m = project_points_to_polyline(lat, lon, base_df[["Lat","Long"]])

        gv["along_m_base"] = along_m
        gv["xtrack_km_base"] = xtrk_m / 1000.0
        gv["p_base"] = p_base
        gv["base_total_km"] = base_total_m / 1000.0

        # coverage stats
        ok = np.isfinite(gv["p_base"].to_numpy(float))
        if ok.sum() >= 2:
            pmin = float(np.nanmin(gv.loc[ok, "p_base"]))
            pmax = float(np.nanmax(gv.loc[ok, "p_base"]))
            cov = pmax - pmin
        else:
            pmin = np.nan; pmax = np.nan; cov = 0.0

        voy_cov_rows.append({
            "voyage_id": int(vid),
            "origin_port_id": str(o),
            "dest_port_id": str(d),
            "n_points_proj": int(ok.sum()),
            "pmin": pmin,
            "pmax": pmax,
            "coverage": float(cov),
            "base_total_km": float(gv["base_total_km"].iloc[0]) if "base_total_km" in gv.columns else np.nan,
        })

        proj_rows.append(gv)

df_clean_points_on_base = pd.concat(proj_rows, ignore_index=True) if proj_rows else pd.DataFrame()
voy_cov = pd.DataFrame(voy_cov_rows)

print("df_clean_points_on_base:", df_clean_points_on_base.shape)
display(voy_cov.sort_values("coverage").head(10))

# ---- ref pool filtering by coverage + points ----
good_vids = voy_cov[
    (voy_cov["n_points_proj"] >= CFG_PROJECT["min_points"]) &
    (voy_cov["coverage"] >= CFG_PROJECT["min_coverage"])
]["voyage_id"].unique().tolist()

df_clean_points_ref_pool = df_clean_points_on_base[df_clean_points_on_base["voyage_id"].isin(good_vids)].copy()

print("ref_pool voyages:", len(good_vids))
print("df_clean_points_ref_pool:", df_clean_points_ref_pool.shape)


df_clean_points_on_base: (24988, 65)


,voyage_id,origin_port_id,dest_port_id,n_points_proj,pmin,pmax,coverage,base_total_km
61,26,TWTPE,TWKEL,108,0.0,0.889290,0.889290,62.624666
39,29,TWTPE,CNFOC,319,0.0,0.981188,0.981188,233.550665
48,67,TWTPE,CNFOC,321,0.0,0.981512,0.981512,233.550665
46,60,TWTPE,CNFOC,301,0.0,0.981517,0.981517,233.550665
35,14,TWTPE,CNFOC,339,0.0,0.981551,0.981551,233.550665
49,76,TWTPE,CNFOC,288,0.0,0.981615,0.981615,233.550665
50,80,TWTPE,CNFOC,284,0.0,0.981634,0.981634,233.550665
43,47,TWTPE,CNFOC,282,0.0,0.981659,0.981659,233.550665
37,21,TWTPE,CNFOC,279,0.0,0.981660,0.981660,233.550665
42,45,TWTPE,CNFOC,273,0.0,0.981736,0.981736,233.550665


ref_pool voyages: 63
df_clean_points_ref_pool: (24948, 65)


### p base to robust, ref route

In [ ]:
REF_CFG = dict(
    n_grid=200,
    min_coverage_abs=2,
    min_coverage_frac=0.30,
)

def _unwrap_lon_deg(lon_deg: np.ndarray) -> np.ndarray:
    lon = np.asarray(lon_deg, dtype=float)
    if len(lon) == 0:
        return lon
    rad = np.deg2rad(lon)
    return np.rad2deg(np.unwrap(rad))

def _interp_with_nan(p_src, x_src, p_grid):
    p_src = np.asarray(p_src, dtype=float)
    x_src = np.asarray(x_src, dtype=float)
    ok = np.isfinite(p_src) & np.isfinite(x_src)
    if ok.sum() < 2:
        return np.full_like(p_grid, np.nan, dtype=float)

    p = p_src[ok]
    x = x_src[ok]
    order = np.argsort(p)
    p = p[order]
    x = x[order]

    # drop duplicate p (keep first unique)
    p_u, idx = np.unique(p, return_index=True)
    x_u = x[idx]
    if len(p_u) < 2:
        return np.full_like(p_grid, np.nan, dtype=float)

    xi = np.interp(p_grid, p_u, x_u)
    xi[(p_grid < p_u.min()) | (p_grid > p_u.max())] = np.nan
    return xi

p_grid = np.linspace(0.0, 1.0, int(REF_CFG["n_grid"]))

ref_rows = []
ref_polylines = {}

if df_clean_points_ref_pool.empty:
    ref_routes_df = pd.DataFrame(columns=["origin_port_id","dest_port_id","n_voyages_used","ref_select_reason"])
else:
    for (o, d), g_od in df_clean_points_ref_pool.groupby(["origin_port_id","dest_port_id"]):
        vids = sorted(g_od["voyage_id"].dropna().unique().tolist())
        if len(vids) == 0:
            continue

        lat_mat = []
        lon_mat = []

        for vid in vids:
            gv = g_od[g_od["voyage_id"] == vid].sort_values("Timestamp").copy()
            if len(gv) < 2:
                continue

            p = pd.to_numeric(gv["p_base"], errors="coerce").to_numpy(float)
            lat = pd.to_numeric(gv["Lat"], errors="coerce").to_numpy(float)
            lon = pd.to_numeric(gv["Long"], errors="coerce").to_numpy(float)

            lon_un = _unwrap_lon_deg(lon)
            lat_i = _interp_with_nan(p, lat, p_grid)
            lon_i = _interp_with_nan(p, lon_un, p_grid)

            lat_mat.append(lat_i)
            lon_mat.append(lon_i)

        if len(lat_mat) == 0:
            continue

        lat_mat = np.vstack(lat_mat)
        lon_mat = np.vstack(lon_mat)

        cov = np.sum(np.isfinite(lat_mat) & np.isfinite(lon_mat), axis=0)
        cov_th = max(int(REF_CFG["min_coverage_abs"]),
                     int(np.ceil(REF_CFG["min_coverage_frac"] * lat_mat.shape[0])))

        lat_ref = np.nanmedian(lat_mat, axis=0)
        lon_ref = np.nanmedian(lon_mat, axis=0)

        lat_ref[cov < cov_th] = np.nan
        lon_ref[cov < cov_th] = np.nan

        ref_df = pd.DataFrame({"p": p_grid, "Lat": lat_ref, "Long": lon_ref, "coverage_n": cov})
        ref_df = ref_df.dropna(subset=["Lat","Long"]).reset_index(drop=True)

        if len(ref_df) < 2:
            ref_rows.append({
                "origin_port_id": str(o), "dest_port_id": str(d),
                "n_voyages_used": int(lat_mat.shape[0]),
                "coverage_threshold": int(cov_th),
                "ref_select_reason": "model_ref_failed_sparse"
            })
            continue

        ref_polylines[(str(o), str(d))] = ref_df
        ref_rows.append({
            "origin_port_id": str(o), "dest_port_id": str(d),
            "n_voyages_used": int(lat_mat.shape[0]),
            "coverage_threshold": int(cov_th),
            "ref_select_reason": "model_ref_from_baseline_p"
        })

    ref_routes_df = pd.DataFrame(ref_rows).sort_values(["origin_port_id","dest_port_id"]).reset_index(drop=True)


print("Built model ref ODs:", len(ref_routes_df))
print("ref_polylines keys:", len(ref_polylines))
display(ref_routes_df.head(10))


Built model ref ODs: 6
ref_polylines keys: 6


C:\Users\slab\AppData\Local\Temp\ipykernel_13168\413873679.py:79: RuntimeWarning: All-NaN slice encountered
  lat_ref = np.nanmedian(lat_mat, axis=0)
C:\Users\slab\AppData\Local\Temp\ipykernel_13168\413873679.py:80: RuntimeWarning: All-NaN slice encountered
  lon_ref = np.nanmedian(lon_mat, axis=0)
C:\Users\slab\AppData\Local\Temp\ipykernel_13168\413873679.py:79: RuntimeWarning: All-NaN slice encountered
  lat_ref = np.nanmedian(lat_mat, axis=0)
C:\Users\slab\AppData\Local\Temp\ipykernel_13168\413873679.py:80: RuntimeWarning: All-NaN slice encountered
  lon_ref = np.nanmedian(lon_mat, axis=0)
C:\Users\slab\AppData\Local\Temp\ipykernel_13168\413873679.py:79: RuntimeWarning: All-NaN slice encountered
  lat_ref = np.nanmedian(lat_mat, axis=0)
C:\Users\slab\AppData\Local\Temp\ipykernel_13168\413873679.py:80: RuntimeWarning: All-NaN slice encountered
  lon_ref = np.nanmedian(lon_mat, axis=0)
C:\Users\slab\AppData\Local\Temp\ipykernel_13168\413873679.py:79: RuntimeWarning: All-NaN slice enco

,origin_port_id,dest_port_id,n_voyages_used,coverage_threshold,ref_select_reason
0,CNFOC,TWTPE,22,7,model_ref_from_baseline_p
1,CNLYA,TWTPE,8,3,model_ref_from_baseline_p
2,JPNGO,TWTPE,3,2,model_ref_from_baseline_p
3,TWTPE,CNFOC,20,6,model_ref_from_baseline_p
4,TWTPE,CNLYA,7,3,model_ref_from_baseline_p
5,TWTPE,TWKEL,3,2,model_ref_from_baseline_p


### Generate ETA training data

### Folium baseline and ref

In [27]:
import folium
from folium import LayerControl
from branca.element import Element

def visualize_baseline_and_ref(
    baselines_by_od: dict,
    ref_polylines: dict,
    ports_df: pd.DataFrame,
    od_list=None,
    output_html="outputs/baseline_vs_ref.html",
    zoom_start=6,
):
    port_map = {str(r["port_id"]):(float(r["lat"]), float(r["lon"])) for _, r in ports_df[["port_id","lat","lon"]].dropna().iterrows()}

    keys = od_list if od_list is not None else list(baselines_by_od.keys())[:10]
    keys = [(str(o), str(d)) for (o, d) in keys]

    # center
    for k in keys:
        if k in baselines_by_od and len(baselines_by_od[k]) >= 2:
            c_lat = float(np.nanmedian(baselines_by_od[k]["Lat"]))
            c_lon = float(np.nanmedian(baselines_by_od[k]["Long"]))
            break
    else:
        print("[WARN] no drawable baseline")
        return None

    m = folium.Map(location=[c_lat, c_lon], zoom_start=zoom_start, tiles="OpenStreetMap")

    for (o, d) in keys:
        if (o, d) in baselines_by_od:
            b = baselines_by_od[(o, d)].dropna(subset=["Lat","Long"])
            coords = list(zip(b["Lat"].astype(float), b["Long"].astype(float)))
            fg = folium.FeatureGroup(name=f"BASE {o}->{d}")
            folium.PolyLine(coords, weight=4, opacity=0.9).add_to(fg)
            fg.add_to(m)

        if (o, d) in ref_polylines:
            r = ref_polylines[(o, d)].dropna(subset=["Lat","Long"])
            coords = list(zip(r["Lat"].astype(float), r["Long"].astype(float)))
            fg = folium.FeatureGroup(name=f"REF  {o}->{d} (model)")
            folium.PolyLine(coords, weight=6, opacity=0.7).add_to(fg)
            fg.add_to(m)

        # port markers
        if o in port_map:
            folium.CircleMarker(port_map[o], radius=5, color="green", fill=True, fill_opacity=0.9,
                                tooltip=f"Origin {o}").add_to(m)
        if d in port_map:
            folium.CircleMarker(port_map[d], radius=5, color="black", fill=True, fill_opacity=0.9,
                                tooltip=f"Dest {d}").add_to(m)

    LayerControl(collapsed=False).add_to(m)

    from pathlib import Path
    outp = Path(output_html)
    outp.parent.mkdir(parents=True, exist_ok=True)
    m.save(str(outp))
    print("saved:", outp.resolve())
    return m

# example:
# visualize_baseline_and_ref(baselines_by_od, ref_polylines, ports_df, od_list=[("CNFOC","TWTPE")], zoom_start=7)


In [28]:
visualize_baseline_and_ref(baselines_by_od, ref_polylines, ports_df, od_list=[("CNFOC","TWTPE")], zoom_start=7)


saved: C:\Users\slab\Desktop\Slab Project\Stage2 ETA\outputs\baseline_vs_ref.html
